# Classroom Compass — Notebook 03: Insights Generation

Runs the full analytical pipeline on the enriched corpus, from TF-IDF and NMF
topic discovery through LLM synthesis, metadata-aware structured extraction,
verification, packaging, and DOCX report generation.

**Run order:** `01_token_preprocess` → `02_semantic_enrichment` → `03_insights_generation`

| Step | What | Key outputs |
|------|------|-------------|
| 1 | Build TF-IDF matrices | in-memory for Steps 2-4 |
| 2 | Quality checkpoint 2 | `quality/quality_cp2.json` |
| 3 | Category TF-IDF | `analysis/category_tfidf.csv` |
| 4 | NMF topic discovery | `analysis/nmf_topics.csv`, `analysis/project_topic_bridge.csv` |
| 5 | LLM topic labeling | `analysis/llm_topic_labels.json` |
| 6 | Synthesis | `analysis/llm_synthesis_*.txt` |
| 7 | Structured insight extraction with optional metadata scope signals | `insights/insights_candidates*.json`, `insights/metadata_lift_facts_for_step7.csv` |
| 8 | Topic verification | updates insights in-memory |
| 9 | Evidence tables | `insights/insights_flat.csv`, `insights/insight_topic_support_candidates.csv` |
| 10 | Packaging, dedupe & topline | updates curated insights in-memory |
| 11 | Final outputs & manifests | `reports/insights_report.docx`, `insights/insights_structured.json` |

Strategic-loop mode is optional. When enabled in `params.yaml`, this notebook
runs one resolved strategic area × split at a time. Existing NB03 filters still
define the corpus scope; strategic-loop preparation happens only after those
filters are applied.


---
## Setup and Configuration

In [1]:
import sys
import json
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from copy import deepcopy

import pandas as pd
import numpy as np
import math
import yaml
import re
import time

ROOT = Path(".")
sys.path.insert(0, str(ROOT))

# Prefer a local utils.py when present. Otherwise load the most recently
# modified utils*.py file so versioned development utilities can be used
# without relying on filename sort order.
import importlib.util as _ilu, sys as _sys
if (ROOT / "utils.py").exists():
    _u = ROOT / "utils.py"
else:
    _utils_candidates = sorted(
        Path(".").glob("utils*.py"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not _utils_candidates:
        raise FileNotFoundError("No utils.py or utils*.py file found in project root.")
    _u = _utils_candidates[0]

if _u.stem != "utils":
    _s = _ilu.spec_from_file_location("utils", _u)
    _m = _ilu.module_from_spec(_s)
    _s.loader.exec_module(_m)
    _sys.modules["utils"] = _m

from utils import (
    load_cfg,
    write_json,
    artifact_meta,
    build_run_output_path,
    get_run_date,
    canonicalize_filter_spec,
    get_filter_fields_key,
    get_run_id,
    apply_filters,
    ensure_warning_file,
    append_warning,
    get_llm_client,
    chat_completion_with_tier_fallback,
    start_stage_manifest,
    finalize_stage_manifest,
    build_pipeline_manifest,
    tokens_to_str,
    make_vec,
    upweight_injected_tokens,
    group_key,
    build_project_topic_bridge,
    quality_report,
    cat_tfidf_slice,
    nmf_one,
    build_input,
    _label_with_retry,
    build_cluster_input,
    _label_cluster_with_retry,
    coerce_token_list,
    _norm_group_value,
    _safe_topic_id,
    build_topic_lines,
    _call_with_retry,
    synthesize_one_group,
    slugify_group_value,
    strip_json_fences,
    normalize_insight,
    build_bridge_lookup,
    build_label_index,
    build_verified_insight_tables,
    apply_deterministic_packaging,
    dedupe_packaged_insights,
    assign_topline_sections_simple,
    build_structured_from_curated,
    build_looker_project_url,
    build_packaged_report_docx,
    project_insight_for_saved_candidates,
    _verify_insight_list,
    resolve_params_path,
    # Strategic-loop helpers
    prepare_strategic_loop_run_dataframe,
    add_strategic_derived_fields,
    expand_strategic_run_plan,
    candidate_support_project_ids_from_source_topics,
    build_metadata_lift_context,
    compute_metadata_lift,
    format_metadata_lift_facts,
    DEFAULT_METADATA_LIFT_DIMENSIONS,
    DEFAULT_METADATA_LIFT_THRESHOLDS,
)

CFG_PATH = resolve_params_path()
CFG = load_cfg(CFG_PATH)

try:
    MANIFEST_CFG_PATH = str(CFG_PATH.relative_to(ROOT))
except ValueError:
    MANIFEST_CFG_PATH = str(CFG_PATH)

ct = CFG["tfidf"]
client = get_llm_client()

# Load the enriched corpus produced by NB02.
raw_df = pd.read_parquet(ROOT / "OUTPUTS/prepared/06_enriched.parquet")

# Stopwords for the quality gate at Step 2. Loaded from params.yaml so the
# list can be extended without touching notebook code.
STOPWORDS = CFG["quality"]["stopword_violation_list"]

print(f"Loaded {len(raw_df):,} rows | {raw_df['project_id'].nunique():,} projects | {raw_df.shape[1]} columns")
print(f"Config path: {CFG_PATH}")
print(f"Utils path: {_u}")


Loaded 2,180,749 rows | 2,175,476 projects | 188 columns
Config path: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/params_v3.yaml
Utils path: utils_v1.4.py


---
## Parameters

All runtime configuration is resolved here from `params.yaml`. Edit
`params.yaml` to change behaviour; do not hardcode values in downstream cells.

Strategic-loop mode is intentionally small in `params.yaml`. The notebook expects
one resolved run at a time, for example:

```yaml
strategic_loop:
  enabled: true
  strategic_area_id: mental_health_sel
  split: strategic_injected_tag
  strategic_areas_path: CONFIG/strategic_areas.yaml
  state_clusters_path: CONFIG/state_clusters.yaml
```

No `run_scope` is used here. Existing NB03 filters define the analysis corpus.


In [2]:
# ── Optional external strategic-loop config loading ──────────────────────────
# Strategic area definitions can live directly inside params.yaml, or in external
# YAML files referenced by strategic_loop.strategic_areas_path and
# strategic_loop.state_clusters_path. This keeps params small while still
# allowing fully resolved single-run configs.

def _read_yaml_if_present(path_value):
    if not path_value:
        return {}
    path = Path(path_value)
    if not path.is_absolute():
        path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(f"Configured YAML path does not exist: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

CFG_RUNTIME = deepcopy(CFG)
STRATEGIC_LOOP_CFG = CFG_RUNTIME.get("strategic_loop", {}) or {}
STRATEGIC_LOOP_ENABLED = bool(STRATEGIC_LOOP_CFG.get("enabled", False))

if STRATEGIC_LOOP_ENABLED:
    strategic_areas_external = _read_yaml_if_present(
        STRATEGIC_LOOP_CFG.get("strategic_areas_path")
    )
    if "strategic_areas" in strategic_areas_external:
        CFG_RUNTIME["strategic_areas"] = strategic_areas_external["strategic_areas"]

    state_clusters_path = STRATEGIC_LOOP_CFG.get("state_clusters_path")
    state_clusters_external = _read_yaml_if_present(state_clusters_path)
    if state_clusters_path and "state_clusters" not in state_clusters_external:
        print(
            "WARNING: strategic_loop.state_clusters_path was provided, but the YAML "
            "does not contain a top-level 'state_clusters:' key. Falling back to "
            "utils.DEFAULT_STATE_CLUSTERS."
        )
    STATE_CLUSTERS = state_clusters_external.get("state_clusters")
else:
    STATE_CLUSTERS = None

print(f"Strategic loop enabled: {STRATEGIC_LOOP_ENABLED}")
if STRATEGIC_LOOP_ENABLED:
    print(f"Strategic area: {STRATEGIC_LOOP_CFG.get('strategic_area_id')}")
    print(f"Split: {STRATEGIC_LOOP_CFG.get('split')}")


Strategic loop enabled: True
Strategic area: stem
Split: strategic_injected_tag


In [19]:
# ── Analysis scope ────────────────────────────────────────────────────────────
BASE_GROUPBY_FIELD = CFG["analysis"]["group_by"]
FILTER_LOGIC       = CFG["analysis"].get("filter_logic", "and")
FILTERS            = CFG["analysis"].get("filters", [])
EXCLUDE_GROUPS     = CFG["analysis"]["exclude_groups"]
REVIEW_GROUP       = CFG["analysis"]["review_group"]

# Apply declarative NB03 filters first. Strategic-loop prep happens after this,
# so no separate strategic run_scope/date filter exists.
df, filter_summary = apply_filters(raw_df, FILTER_LOGIC, FILTERS)
if filter_summary["no_rows_after_filter"]:
    raise ValueError("No rows remain after applying analysis filters. Check params.yaml.")

BASE_FILTERED_PROJECT_COUNT = int(df["project_id"].nunique())

# ── v1 feature guards ────────────────────────────────────────────────────────
# Binned/time-slice mode is still guarded out because topic identity and
# source_topic traceability are keyed by group + topic_id only.
if CFG["analysis"].get("bins", []):
    raise ValueError(
        "analysis.bins is non-empty: binned/time-slice mode is not supported "
        "in v1. Topic identity and source_topic traceability are not bin-aware. "
        "Set analysis.bins: [] to run without time bucketing."
    )

if REVIEW_GROUP is not None:
    raise ValueError(
        f"analysis.review_group is set to {REVIEW_GROUP!r}: single-group review "
        "mode is not supported in v1. Set analysis.review_group: null to run "
        "against all groups."
    )

# Legacy waterfall fields from older NB03 versions were derived in-notebook.
# v1.5 does not recreate them in legacy mode; use strategic_loop instead.
_REMOVED_LEGACY_DERIVED_GROUPBY_FIELDS = {
    "funding_status_x_metro",
    "efs_x_grade_band",
    "specialty_subject",
    "industry",
    "safety_justice",
    "future_framing",
    "framing_context",
    "current_pressures",
}
if not STRATEGIC_LOOP_ENABLED and BASE_GROUPBY_FIELD in _REMOVED_LEGACY_DERIVED_GROUPBY_FIELDS:
    raise ValueError(
        f"analysis.group_by={BASE_GROUPBY_FIELD!r} was a derived grouping used in older NB03 versions. "
        "Enable strategic_loop with the equivalent strategic area/split, or change analysis.group_by "
        "to a physical column in OUTPUTS/prepared/06_enriched.parquet."
    )

# Add the same standard derived fields in legacy mode that strategic-mode prep adds.
# This preserves v1.5.1 metadata-lift behavior and lets legacy runs group on
# physical-or-standard-derived columns such as project_cost_bucket or state_cluster.
if not STRATEGIC_LOOP_ENABLED:
    df = add_strategic_derived_fields(df, state_clusters=STATE_CLUSTERS)

if not STRATEGIC_LOOP_ENABLED and BASE_GROUPBY_FIELD not in df.columns:
    raise ValueError(
        f"analysis.group_by={BASE_GROUPBY_FIELD!r} is not present in the filtered enriched dataframe. "
        "Use a physical column or enable strategic_loop with a resolved split."
    )

def _validate_selected_area_tag_columns(cfg_runtime, strategic_area_id, frame, *, allow_missing=False):
    """Validate selected-area tag_* schema before strategic prep.

    Strict mode is the default. allow_missing=True lets exploratory runs proceed
    after dropping missing tags from this run's area spec, but still fails if no
    tags remain available.
    """
    strategic_areas = cfg_runtime.get("strategic_areas", {}) or {}
    if strategic_area_id not in strategic_areas:
        raise ValueError(f"Unknown strategic_area_id {strategic_area_id!r} in strategic_loop config.")

    area_spec = deepcopy(strategic_areas[strategic_area_id] or {})
    include_tags = area_spec.get("include_taxonomy_tags", []) or []
    if not include_tags:
        raise ValueError(
            f"Strategic area {strategic_area_id!r} has no include_taxonomy_tags. "
            "NB03 strategic runs currently require tag_* boolean membership columns."
        )

    available_tags = [tag for tag in include_tags if f"tag_{tag}" in frame.columns]
    missing_cols = [f"tag_{tag}" for tag in include_tags if f"tag_{tag}" not in frame.columns]

    if missing_cols and not allow_missing:
        raise ValueError(
            f"Strategic area {strategic_area_id!r} is missing {len(missing_cols)} required tag columns "
            f"in 06_enriched.parquet. Example missing columns: {missing_cols[:10]}. "
            "Re-run NB02 with the current taxonomy, update strategic_areas.yaml, or set "
            "strategic_loop.allow_missing_tag_columns: true for exploratory soft-fail mode."
        )

    if missing_cols and allow_missing:
        print(
            f"WARNING: Strategic area {strategic_area_id!r} is missing {len(missing_cols)} tag columns; "
            f"dropping them for this run because strategic_loop.allow_missing_tag_columns=true. "
            f"Example missing columns: {missing_cols[:10]}"
        )
        if not available_tags:
            raise ValueError(
                f"Strategic area {strategic_area_id!r} has no available tag_* columns after dropping missing tags."
            )
        area_spec["include_taxonomy_tags"] = available_tags

    return area_spec

# ── Strategic-loop dataframe preparation ─────────────────────────────────────
# The prep helper handles:
# - membership from tag_{taxonomy_tag} columns
# - derived fields: state_cluster, project_cost_bucket, funding_status, posting_period
# - project_category_bucketed with Missing/Awaiting Classification -> All Other
# - strategic_injected_tag explosion
# - 200-project hard group minimum, unless overridden in config
# - removal of area-defining injected tokens for strategic_injected_tag only
if STRATEGIC_LOOP_ENABLED:
    strategic_area_id = STRATEGIC_LOOP_CFG.get("strategic_area_id")
    split_spec = STRATEGIC_LOOP_CFG.get("split")
    if not strategic_area_id or split_spec is None:
        raise ValueError(
            "strategic_loop.enabled is true, but strategic_area_id or split is missing."
        )

    area_spec = _validate_selected_area_tag_columns(
        CFG_RUNTIME,
        strategic_area_id,
        raw_df,
        allow_missing=bool(STRATEGIC_LOOP_CFG.get("allow_missing_tag_columns", False)),
    )

    # Keep NB03 one-run-at-a-time and avoid scanning every strategic area's tag columns.
    CFG_FOR_THIS_RUN = deepcopy(CFG_RUNTIME)
    CFG_FOR_THIS_RUN["strategic_areas"] = {strategic_area_id: area_spec}

    project_category_bucket_rules = (
        STRATEGIC_LOOP_CFG.get("project_category_bucket_rules")
        or CFG_RUNTIME.get("project_category_bucket_rules")
        or {}
    )
    min_group_projects = int(
        STRATEGIC_LOOP_CFG.get(
            "min_projects_per_group",
            STRATEGIC_LOOP_CFG.get(
                "min_projects_per_strategic_injected_tag_group",
                CFG_RUNTIME.get("defaults", {}).get("min_projects_per_strategic_injected_tag_group", 200),
            ),
        )
    )

    df, STRATEGIC_RUN_META = prepare_strategic_loop_run_dataframe(
        df,
        cfg=CFG_FOR_THIS_RUN,
        strategic_area_id=strategic_area_id,
        split_spec=split_spec,
        min_group_projects=min_group_projects,
        project_category_bucket_rules=project_category_bucket_rules,
        state_clusters=STATE_CLUSTERS,
    )
    GROUPBY_FIELD = STRATEGIC_RUN_META["groupby_field"]
else:
    GROUPBY_FIELD = BASE_GROUPBY_FIELD
    STRATEGIC_RUN_META = {
        "strategic_loop_enabled": False,
        "strategic_area_id": None,
        "strategic_area_label": None,
        "split_id": None,
        "groupby_fields": [GROUPBY_FIELD],
        "groupby_field": GROUPBY_FIELD,
        "is_strategic_injected_tag": False,
        "min_group_projects": None,
        "input_project_count": BASE_FILTERED_PROJECT_COUNT,
        "area_project_count": None,
        "run_row_count": int(len(df)),
        "run_project_count": int(df["project_id"].nunique()),
        "group_count": int(df[GROUPBY_FIELD].nunique(dropna=True)),
        "group_counts": [],
        "removed_area_defining_injected_tokens": False,
    }

# ── Group scope: apply exclude_groups once, after the final groupby is resolved ─
if EXCLUDE_GROUPS:
    n_before = len(df)
    df = df[
        ~df[GROUPBY_FIELD].astype(str).isin([str(g) for g in EXCLUDE_GROUPS])
    ].reset_index(drop=True)
    if df.empty:
        raise ValueError(
            f"No rows remain after excluding groups {EXCLUDE_GROUPS}. "
            "Check analysis.exclude_groups in params.yaml."
        )
    print(
        f"exclude_groups: {n_before - len(df):,} rows removed "
        f"({len(EXCLUDE_GROUPS)} group(s): {EXCLUDE_GROUPS})"
    )
else:
    df = df.reset_index(drop=True)

# Refresh meta counts after exclude_groups so printed metadata and manifests match the actual run dataframe.
STRATEGIC_RUN_META["run_row_count"] = int(len(df))
STRATEGIC_RUN_META["run_project_count"] = int(df["project_id"].nunique())
STRATEGIC_RUN_META["group_count"] = int(df[GROUPBY_FIELD].nunique(dropna=True))

_fresh_group_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=[GROUPBY_FIELD, "project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
    .rename("project_count")
    .reset_index()
    .sort_values("project_count", ascending=False)
)

# Preserve group-level audit trails whenever strategic prep supplied one.
# This applies to both strategic_injected_tag and non-tag strategic splits: the
# prep step may include groups dropped by the 200-project floor, and exclude_groups
# may remove groups that originally cleared the floor. Keep those entries visible
# with kept_for_run=False instead of replacing the list with only surviving groups.
_existing_group_counts = STRATEGIC_RUN_META.get("group_counts") or []
if _existing_group_counts:
    _fresh_count_lookup = {
        str(row[GROUPBY_FIELD]): int(row["project_count"])
        for _, row in _fresh_group_counts.iterrows()
    }
    _refreshed_group_counts = []
    for row in _existing_group_counts:
        group_value = row.get(GROUPBY_FIELD, row.get("strategic_injected_tag", row.get("taxonomy_tag")))
        group_value_key = str(group_value)
        new_row = {**row}
        surviving = group_value_key in _fresh_count_lookup
        new_row["kept_for_run"] = bool(surviving)
        if surviving:
            new_row["project_count"] = int(_fresh_count_lookup[group_value_key])
        _refreshed_group_counts.append(new_row)
    STRATEGIC_RUN_META["group_counts"] = _refreshed_group_counts
else:
    _fresh_records = _fresh_group_counts.to_dict(orient="records")
    for row in _fresh_records:
        row["kept_for_run"] = True
    STRATEGIC_RUN_META["group_counts"] = _fresh_records

# ── Run identity ──────────────────────────────────────────────────────────────
FILTER_SPEC = canonicalize_filter_spec(FILTER_LOGIC, FILTERS)
if STRATEGIC_LOOP_ENABLED:
    RUN_SCOPE_SPEC = deepcopy(FILTER_SPEC)
    RUN_SCOPE_SPEC["strategic_loop"] = {
        "enabled": True,
        "strategic_area_id": STRATEGIC_RUN_META.get("strategic_area_id"),
        "split_id": STRATEGIC_RUN_META.get("split_id"),
        "groupby_field": GROUPBY_FIELD,
    }
else:
    # Preserve v1.4 legacy-mode run hashes by hashing only FILTER_SPEC.
    RUN_SCOPE_SPEC = FILTER_SPEC

FILTER_FIELDS_KEY = get_filter_fields_key(FILTERS)
RUN_ID   = get_run_id(GROUPBY_FIELD, RUN_SCOPE_SPEC)
RUN_DATE = get_run_date()

def OUT(subdir, fname):
    return build_run_output_path(
        subdir=subdir,
        fname=fname,
        groupby_field=GROUPBY_FIELD,
        run_id=RUN_ID,
        strategic_loop_enabled=STRATEGIC_LOOP_ENABLED,
        strategic_area_id=STRATEGIC_RUN_META.get("strategic_area_id"),
    )

# ── Analysis description and topic/slice settings ────────────────────────────
if STRATEGIC_LOOP_ENABLED:
    area_label = STRATEGIC_RUN_META.get("strategic_area_label") or STRATEGIC_RUN_META.get("strategic_area_id")
    split_id = STRATEGIC_RUN_META.get("split_id") or GROUPBY_FIELD
    GROUP_DESCRIPTION = (
        f"strategic area '{area_label}' split by {split_id}; "
        f"group values are values of '{GROUPBY_FIELD}'"
    )
else:
    GROUP_DESCRIPTION = CFG["analysis"]["group_descriptions"].get(
        GROUPBY_FIELD,
        f"groups defined by '{GROUPBY_FIELD}' in DonorsChoose data",
    )

MIN_SHARED    = CFG["analysis"]["nmf_min_shared"]
MIN_COVERAGE  = CFG["analysis"]["min_coverage"]
BASE_N_COMPONENTS = CFG["nmf"]["n_components"]
CAT_TFIDF_TOP_N   = CFG["analysis"]["cat_tfidf_top_n"]

SLICE_RULES   = CFG["analysis"]["slice_rules"]
VERIFY_CFG    = CFG["analysis"]["verification"]
PACKAGING_CFG = CFG["analysis"]["packaging"]
DEDUPE_CFG    = CFG["analysis"]["dedupe"]

group_project_counts = (
    df[[GROUPBY_FIELD, "project_id"]]
    .dropna(subset=["project_id"])
    .drop_duplicates()
    .groupby(GROUPBY_FIELD)["project_id"]
    .nunique()
)
median_group_projects = float(group_project_counts.median()) if not group_project_counts.empty else 0.0
SMALL_SLICE_MODE = median_group_projects < SLICE_RULES["small_slice_cutoff"]
SLICE_RULES = {**SLICE_RULES, "small_slice_mode": SMALL_SLICE_MODE}

MIN_GROUP_PROJECTS_EFFECTIVE = max(
    CFG["analysis"]["min_group_projects"],
    SLICE_RULES["min_group_projects"],
)
CROSS_MIN_INSIGHTS     = SLICE_RULES["cross_min_insights"]
CROSS_MAX_INSIGHTS     = SLICE_RULES["cross_max_insights"]
PER_GROUP_MIN_INSIGHTS = SLICE_RULES["per_group_min_insights"]
PER_GROUP_MAX_INSIGHTS = SLICE_RULES["per_group_max_insights"]

# ── LLM settings ──────────────────────────────────────────────────────────────
LLM_CFG                   = CFG.get("llm", {}) or {}
N_REPRESENTATIVE          = LLM_CFG["n_representative_snippets"]
TOP_TERMS_IN_PROMPT       = LLM_CFG["top_terms_in_prompt"]
SYNTHESIS_TOP_TERMS_COUNT = LLM_CFG["synthesis_top_terms_count"]

MAX_RETRIES               = int(LLM_CFG.get("max_retries", 3))
LLM_SERVICE_TIER          = str(LLM_CFG.get("service_tier", "flex"))
LLM_FLEX_ATTEMPTS         = int(LLM_CFG.get("flex_attempts", 2))
LLM_FALLBACK_SERVICE_TIER = str(LLM_CFG.get("fallback_service_tier", "default"))
LLM_FLEX_RETRY_DELAY      = float(LLM_CFG.get("flex_retry_delay_seconds", 0))
LLM_RETRY_DELAY           = float(LLM_CFG.get("retry_delay_seconds", 1))
LLM_TIMEOUT_SECONDS       = float(LLM_CFG.get("timeout_seconds", 900))

MODEL_LABELING  = CFG["models"]["labeling"]
MODEL_SYNTHESIS = CFG["models"]["synthesis"]
MODEL_VERIFY    = CFG["models"]["verify"]

MAX_WORKERS          = CFG["analysis"]["synthesis_max_workers"]
LABELING_MAX_WORKERS = CFG["analysis"]["labeling_max_workers"]

# ── Metadata lift settings for Step 7 ─────────────────────────────────────────
METADATA_LIFT_CFG = CFG.get("metadata_lift", {}) or {}
METADATA_LIFT_ENABLED = bool(METADATA_LIFT_CFG.get("enabled", True))
if "min_lift" in METADATA_LIFT_CFG:
    print(
        "WARNING: metadata_lift.min_lift is deprecated and ignored. "
        "Use metadata_lift.min_cohens_h instead. Lift is still reported in the prompt "
        "after metadata facts are selected."
    )
METADATA_LIFT_DIMENSIONS = METADATA_LIFT_CFG.get("dimensions", DEFAULT_METADATA_LIFT_DIMENSIONS)
METADATA_LIFT_THRESHOLDS = {
    **DEFAULT_METADATA_LIFT_THRESHOLDS,
    **{k: v for k, v in METADATA_LIFT_CFG.items() if k in DEFAULT_METADATA_LIFT_THRESHOLDS},
}

# ── Output settings ──────────────────────────────────────────────────────────
LOOKER_BASE_URL     = CFG["output"]["looker_base_url"]
LOOKER_FILTER_FIELD = CFG["output"]["looker_filter_field"]
LOOKER_FIELDS       = CFG["output"]["looker_fields"]
LOOKER_LIMIT        = int(CFG["output"].get("looker_limit", 500))
LOOKER_ID_LIMIT     = int(CFG["output"].get("looker_id_limit", 100))
CSV_MAX_IDS_PER_INSIGHT = int(CFG["output"]["csv_max_ids_per_insight"])
MAIN_MIN_VERIFICATION_RATIO = PACKAGING_CFG["main_min_verification_ratio"]

# bins are guarded out above; group_cols is always [GROUPBY_FIELD] in v1.
group_cols = [GROUPBY_FIELD]

# ── Warnings, filter provenance, stage manifest ──────────────────────────────
WARNINGS_PATH       = OUT("metadata", "warnings_03.jsonl")
FILTER_SPEC_PATH    = OUT("metadata", "filter_spec.json")
FILTER_SUMMARY_PATH = OUT("metadata", "filter_summary.json")
STRATEGIC_META_PATH = OUT("metadata", "strategic_run_meta.json")
COPIED_CONFIG_PATH  = OUT("metadata", CFG_PATH.name)

ensure_warning_file(WARNINGS_PATH)

filter_spec_payload = {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    "filter_fields_key": FILTER_FIELDS_KEY,
    "filter_logic": FILTER_LOGIC,
    "filters": FILTERS,
}
if STRATEGIC_LOOP_ENABLED:
    filter_spec_payload["strategic_loop"] = RUN_SCOPE_SPEC["strategic_loop"]
write_json(FILTER_SPEC_PATH, filter_spec_payload)

write_json(FILTER_SUMMARY_PATH, {
    "schema_version": "v1",
    "run_id": RUN_ID,
    "group_by_field": GROUPBY_FIELD,
    **filter_summary,
})
write_json(STRATEGIC_META_PATH, {
    "schema_version": "v1",
    "run_id": RUN_ID,
    **STRATEGIC_RUN_META,
})
shutil.copy2(CFG_PATH, COPIED_CONFIG_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="03_insights_generation",
    notebook_file="03_insights_generation_v1.5.4_strategic_loop.ipynb",
    config_path=MANIFEST_CFG_PATH,
    run_id=RUN_ID,
    group_by_field=GROUPBY_FIELD,
    filter_fields_key=FILTER_FIELDS_KEY,
)

print(f"RUN_ID            = {RUN_ID}")
print(f"GROUPBY_FIELD     = {GROUPBY_FIELD!r}")
print(f"FILTER_FIELDS_KEY = {FILTER_FIELDS_KEY}")
print(f"Filtered projects before strategic prep = {BASE_FILTERED_PROJECT_COUNT:,}")
print(f"Run rows          = {len(df):,}")
print(f"Run projects      = {df['project_id'].nunique():,}")
print(f"Run groups        = {df[GROUPBY_FIELD].nunique(dropna=True):,}")
print(f"small_slice_mode  = {SMALL_SLICE_MODE} (median group = {median_group_projects:.1f})")
if STRATEGIC_LOOP_ENABLED:
    print(f"Output root       = OUTPUTS/runs/strategic_area/{STRATEGIC_RUN_META.get('strategic_area_id')}/{RUN_ID}/")
else:
    print(f"Output root       = OUTPUTS/runs/non_strategic/{GROUPBY_FIELD}/{RUN_ID}/")

if STRATEGIC_LOOP_ENABLED:
    print("\nStrategic run summary:")
    for key in [
        "strategic_area_id",
        "strategic_area_label",
        "split_id",
        "groupby_fields",
        "is_strategic_injected_tag",
        "min_group_projects",
        "input_project_count",
        "area_project_count",
        "run_project_count",
        "group_count",
        "removed_area_defining_injected_tokens",
    ]:
        print(f"  {key:40s} = {STRATEGIC_RUN_META.get(key)}")


exclude_groups: 0 rows removed (1 group(s): ['__NA__'])
RUN_ID            = 20260522_160015_strategic_injected_tag_b09019b8
GROUPBY_FIELD     = 'strategic_injected_tag'
FILTER_FIELDS_KEY = expiration_date
Filtered projects before strategic prep = 705,785
Run rows          = 112,480
Run projects      = 92,077
Run groups        = 9
small_slice_mode  = False (median group = 11435.0)
Output root       = OUTPUTS/runs/strategic_area/stem/20260522_160015_strategic_injected_tag_b09019b8/

Strategic run summary:
  strategic_area_id                        = stem
  strategic_area_label                     = STEM
  split_id                                 = strategic_injected_tag
  groupby_fields                           = ['strategic_injected_tag']
  is_strategic_injected_tag                = True
  min_group_projects                       = 200
  input_project_count                      = 705785
  area_project_count                       = 92077
  run_project_count                        = 9207

---
## Step 1 — Build TF-IDF Matrices

Fits one TF-IDF vectorizer per n-gram range on the full filtered corpus and
keeps the resulting sparse matrices in memory. All downstream steps reuse these
objects rather than refitting.

Trigrams are built when `ngrams.max_n >= 3` in `params.yaml`.

In [5]:
docs = df["tokens"].apply(tokens_to_str).tolist()

# Build one matrix per n-gram range. X_unigram_bigram is the primary matrix
# for category TF-IDF and NMF; the others are retained for debugging and
# future analytical passes.
specs = {
    "X_unigram":        (1, 1),
    "X_bigram":         (2, 2),
    "X_unigram_bigram": (1, 2),
}
if CFG["ngrams"]["max_n"] >= 3:
    specs["X_trigram"] = (3, 3)

matrices, vecs = {}, {}
for name, rng in specs.items():
    vec = make_vec(ct["min_df"], ct["max_df"], rng)
    matrices[name] = vec.fit_transform(docs)
    vecs[name] = vec
    sz, nnz = matrices[name].shape, matrices[name].nnz
    print(f"  {name:20s}: shape={sz}  sparsity={1 - nnz / (sz[0] * sz[1]):.3f}")

# Spot-check first features to catch bad filtering or token drift.
for name, vec in vecs.items():
    print(f"  {name:20s} first feats → {vec.get_feature_names_out()[:10].tolist()}")

  X_unigram           : shape=(112480, 6480)  sparsity=0.992
  X_bigram            : shape=(112480, 118239)  sparsity=1.000
  X_unigram_bigram    : shape=(112480, 124719)  sparsity=0.999
  X_trigram           : shape=(112480, 36459)  sparsity=1.000
  X_unigram            first feats → ['aac', 'aba', 'abandon', 'abc', 'abcs', 'ability', 'abound', 'above', 'absence', 'absent']
  X_bigram             first feats → ['aac communicate', 'aac device', 'aba practice', 'abc number', 'ability abstract', 'ability academic', 'ability access', 'ability accessible', 'ability achieve', 'ability acquire']
  X_unigram_bigram     first feats → ['aac', 'aac communicate', 'aac device', 'aba', 'aba practice', 'abandon', 'abc', 'abc number', 'abcs', 'ability']
  X_trigram            first feats → ['aac communicate world', 'aba practice staff', 'ability actively participate', 'ability add subtract', 'ability allow differentiate', 'ability analyze complex', 'ability analyze datum', 'ability analyze text', 'ab

---
## Step 2 — Quality Checkpoint 2

Gate before LLM calls. Review before proceeding:

- **No stopword violations** — if terms from `quality.stopword_violation_list`
  appear in the top-200 vocab, re-run NB01 with tighter frequency thresholds.
- **Reasonable token distribution** — `p50` should be in the 20–60 range.
- **Matrix sparsity** — very high sparsity on the bigram matrix suggests
  `tfidf.min_df` may be too restrictive.

In [6]:
# Pass the configured stopword list so the gate reflects params.yaml rather
# than the HARD_STOPWORDS fallback in utils.py.
qr2 = quality_report(
    df, label="cp2",
    matrices=matrices,
    save_path=OUT("quality", "quality_cp2.json"),
    stopwords=STOPWORDS,
)


=======================================================  [cp2]
  Projects : 112,480
  Tok/proj : min=11  p50=53  max=96
  Vocab    : 7,402 unique tokens
  X_unigram           : shape=[112480, 6480]  sparsity=0.992
  X_bigram            : shape=[112480, 118239]  sparsity=1.000
  X_unigram_bigram    : shape=[112480, 124719]  sparsity=0.999
  X_trigram           : shape=[112480, 36459]  sparsity=1.000
  Stopwords: PASS



---
## Step 3 — Category TF-IDF

The vectorizer is fit **once** on the full corpus; category slices are scored
by index. This avoids the string-comparison bug (identical token sets across
projects would misclassify rows) and is much faster than refitting per slice.

**Contrast** = token prevalence in this category minus prevalence outside it.

Time bins: defined in `params.yaml` under `analysis.bins`; leave empty for
the full date range.


In [7]:
# ── Category TF-IDF ────────────────────────────────────────────────────────
# Score each eligible group against the rest of the corpus using a shared matrix.

top_n = CAT_TFIDF_TOP_N
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

df_work = df.copy().reset_index(drop=True)
all_docs = df_work["tokens"].apply(tokens_to_str).tolist()

vec_cat = make_vec(ct["min_df"], ct["max_df"], tuple(ct["ngram_range"]))
X_full = vec_cat.fit_transform(all_docs)
X_full = upweight_injected_tokens(
    X_full, vec_cat,
    weight=float(ct.get("injected_token_weight", 1.0)),
    renormalize=True,
)
feat = vec_cat.get_feature_names_out()
idf_vals = vec_cat.idf_

rows = []
for keys, sub in df_work.groupby(group_cols, observed=True):
    if len(sub) < min_proj:
        continue

    kd = group_key(keys, group_cols)
    top = cat_tfidf_slice(
        sub.index,
        df_index=df_work.index,
        X_full=X_full,
        feat=feat,
        idf_vals=idf_vals,
        top_n=top_n,
    )
    for col, val in kd.items():
        top.insert(0, col, val)
    rows.append(top)

if not rows:
    raise RuntimeError(
        f"No groups met min_proj={min_proj} threshold — lower min_group_projects in params.yaml"
    )

cat_tfidf_df = pd.concat(rows, ignore_index=True)
cat_tfidf_df.to_csv(OUT("analysis", "category_tfidf.csv"), index=False)

print(f"{len(cat_tfidf_df):,} rows  |  {cat_tfidf_df[group_cols[0]].nunique()} groups")
cat_tfidf_df.head(10)

270 rows  |  9 groups


,strategic_injected_tag,token,tf,idf,tfidf,prevalence,contrast,project_count
0,framing_experiential_hands_on_learning,microscope,0.028475,4.398897,0.125258,0.336276,0.323047,2362
1,framing_experiential_hands_on_learning,simulation,0.023250,4.539218,0.105539,0.301965,0.291117,2121
2,framing_experiential_hands_on_learning,experiential,0.017830,4.867944,0.086796,0.221384,0.213845,1555
3,framing_experiential_hands_on_learning,science kit,0.009342,5.838195,0.054543,0.085421,0.082671,600
4,framing_experiential_hands_on_learning,slide,0.010350,5.245929,0.054295,0.102933,0.094522,723
5,framing_experiential_hands_on_learning,science,0.023033,2.262249,0.052106,0.566344,0.302205,3978
6,framing_experiential_hands_on_learning,cell,0.008939,5.118468,0.045756,0.094533,0.083486,664
7,framing_experiential_hands_on_learning,world,0.016574,2.388439,0.039585,0.413582,0.175056,2905
8,framing_experiential_hands_on_learning,lab,0.011654,3.342590,0.038955,0.196754,0.107390,1382
9,framing_experiential_hands_on_learning,microscopic,0.006031,6.014474,0.036274,0.056236,0.052907,395


---
## Step 4 — NMF Topic Discovery

NMF is fit independently per group so the dominant vocabulary axis in one group
does not suppress signal in others. Topics are treated as evidence candidates
for LLM synthesis — not as stable theme definitions.

The NMF weight matrix `W` records how strongly each project loads on each topic.
Step 5 uses the top-weight projects per topic as representative snippets rather
than sampling randomly.

In [8]:
# ── Groupwise NMF topics ───────────────────────────────────────────────────
# Fit one NMF model per eligible group and keep both topic-level and project-level outputs.

cn = CFG["nmf"]
min_proj = MIN_GROUP_PROJECTS_EFFECTIVE

_itw = float(ct.get("injected_token_weight", 1.0))
if _itw != 1.0:
    print(f"  injected_token_weight = {_itw} (L2-renormalized per row)")

all_topics, all_weights = [], []
df_work = df.copy().reset_index(drop=True)

NMF_GROUPS_SKIPPED = []
NMF_GROUPS_FAILED = []

for keys, sub in df_work.groupby(group_cols, observed=True):
    kd = group_key(keys, group_cols)
    group_value = kd[GROUPBY_FIELD]

    # Skip groups that are too small to support stable topic extraction.
    if len(sub) < min_proj:
        NMF_GROUPS_SKIPPED.append(str(group_value))
        continue

    group_docs = sub["tokens"].apply(tokens_to_str).tolist()
    pids = sub["project_id"].tolist()

    try:
        topics, W, nmf_meta = nmf_one(
            group_docs,
            ct_cfg=ct,
            cn_cfg=cn,
            base_n_components=BASE_N_COMPONENTS,
            slice_rules=SLICE_RULES,
        )
    except Exception as e:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF failed for group '{group_value}'",
            context={"group": group_value, "error": str(e)},
        )
        continue

    # None return means the slice was too thin; kept separate from exceptions for QA.
    if topics is None or W is None:
        NMF_GROUPS_FAILED.append(str(group_value))
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "NMF_GROUP_SKIPPED",
            f"NMF skipped group '{group_value}' because the slice was too thin",
            context={"group": group_value, **(nmf_meta or {})},
        )
        continue

    for col, val in kd.items():
        topics[col] = val
    all_topics.append(topics)

    # Persist the full W ranking so later cells can recover top projects per topic.
    for tid in range(W.shape[1]):
        order = W[:, tid].argsort()[::-1]
        for rank, idx in enumerate(order):
            all_weights.append(
                {
                    **kd,
                    "topic_id": tid,
                    "project_id": pids[idx],
                    "weight": float(W[idx, tid]),
                    "rank": rank,
                }
            )

if not all_topics:
    raise RuntimeError(
        "No groups produced NMF topics — check n_components vs retained vocab, "
        "or lower min_group_projects"
    )

topics_df = pd.concat(all_topics, ignore_index=True)
weights_df = pd.DataFrame(all_weights)

topics_df.to_csv(OUT("analysis", "nmf_topics.csv"), index=False)
weights_df.to_csv(OUT("analysis", "nmf_weights.csv"), index=False)

print(f"{len(topics_df):,} topics across {topics_df[group_cols[0]].nunique()} groups")

# Build the project-topic bridge once so downstream evidence collection can reuse it.
threshold = CFG["analysis"]["topic_assignment_threshold"]
project_topic_bridge_df = build_project_topic_bridge(
    weights_df,
    GROUPBY_FIELD,
    threshold,
)
project_topic_bridge_df.to_csv(OUT("analysis", "project_topic_bridge.csv"), index=False)
print(
    "Bridge: "
    f"{len(project_topic_bridge_df):,} project-topic assignments "
    f"(topic_share >= {threshold})"
)

topics_df.head(6)

  injected_token_weight = 2.0 (L2-renormalized per row)


/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/opt/miniconda3/lib/python3.13/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


135 topics across 9 groups
Bridge: 98,169 project-topic assignments (topic_share >= 0.3)


,topic_id,top_terms,top_weights,strategic_injected_tag
0,0,"[simulation, digital, video, interactive, acce...","[0.997123080050396, 0.8580912934591627, 0.7306...",framing_experiential_hands_on_learning
1,1,"[around, world around, question, world, curios...","[0.6234535231220459, 0.5796259353597294, 0.491...",framing_experiential_hands_on_learning
2,2,"[social, emotional, experiential, social emoti...","[0.7746448663257384, 0.6771044638634724, 0.603...",framing_experiential_hands_on_learning
3,3,"[solve, problem solve, thinke, problem, critic...","[0.8623575432353575, 0.8493501895434059, 0.820...",framing_experiential_hands_on_learning
4,4,"[thank advance, developmental delay, diagnose,...","[0.44665327982258085, 0.43921052720871767, 0.4...",framing_experiential_hands_on_learning
5,5,"[plant, garden, environmental, soil, food, exp...","[0.5212942282111502, 0.5116940584636808, 0.444...",framing_experiential_hands_on_learning


In [9]:
# ── Cross-group universal themes ───────────────────────────────────────────
# Identify term bundles that recur across groups after topic extraction.

# Select the largest group as the cross-group reference point for any
# group-specific context needed in downstream steps.
REFERENCE_GROUP = df_work.groupby(GROUPBY_FIELD).size().idxmax()

records = [(r[group_cols[0]], frozenset(r.top_terms)) for _, r in topics_df.iterrows()]

# theme_cats maps a shared term bundle to the set of groups where it recurs.
theme_cats = defaultdict(set)
for i, (ci, si) in enumerate(records):
    for cj, sj in records[i + 1:]:
        if ci == cj:
            continue
        shared = si & sj
        if len(shared) >= MIN_SHARED:
            key = frozenset(shared)
            theme_cats[key] |= {ci, cj}

rows = sorted(theme_cats.items(), key=lambda x: -len(x[1]))
seen = []
deduped = []
for terms, cats in rows:
    if len(cats) < MIN_COVERAGE:
        continue
    if not any(terms <= prior for prior in seen):
        deduped.append(
            {
                "theme": ", ".join(sorted(terms)[:5]),
                "n_groups": len(cats),
                "categories": sorted(cats),
            }
        )
        seen.append(terms)

cross_group_df = pd.DataFrame(deduped).reset_index(drop=True)
cross_group_df.to_csv(OUT("analysis", "cross_group_themes.csv"), index=False)
cross_group_df

,theme,n_groups,categories
0,"collaboration, critical, critical thinke, deve...",6,"[framing_experiential_hands_on_learning, frami..."
1,"design, engineering, stem",6,"[framing_experiential_hands_on_learning, frami..."
2,"item, material, supply",6,"[framing_experiential_hands_on_learning, frami..."
3,"fine, motor, play",5,"[framing_experiential_hands_on_learning, frami..."
4,"creativity, problem, solve",5,"[framing_experiential_hands_on_learning, frami..."
5,"collaboration, critical, critical thinke, deve...",5,"[framing_experiential_hands_on_learning, frami..."
6,"collaboration, creativity, critical, critical ...",5,"[framing_experiential_hands_on_learning, frami..."
7,"book, love, read",5,"[framing_experiential_hands_on_learning, frami..."
8,"item, keep, material, supply",5,"[framing_inquiry_exploration, framing_programm..."
9,"collaboration, critical, critical thinke, deve...",4,"[framing_experiential_hands_on_learning, frami..."


In [10]:
# ── Topic clustering layer (additive, behind topic_clustering.enabled flag) ──
# Builds analysis units between Step 4 NMF topics and Step 5 LLM labeling.
# Produces:
#   analysis_units_df       — rows are clusters + singletons; this is the new
#                             input shape for Step 5 when enabled
#   cluster_membership_df   — per-cluster member rows with group, topic_id,
#                             distinctives, and cluster_support_score
#   generic_clusters_df     — clusters that failed the concrete-core rule;
#                             saved for audit even when dropped from units
#
# When topic_clustering.enabled is False the cell still runs and writes
# diagnostic CSVs, but does not replace topics_df. Step 5 should branch on
# topic_clustering.enabled.

from collections import Counter, defaultdict
from itertools import combinations

TC_CFG = CFG.get("topic_clustering", {}) or {}
TC_ENABLED          = bool(TC_CFG.get("enabled", False))
TC_JACCARD          = float(TC_CFG.get("jaccard_threshold", 0.50))
TC_TOP_N            = int(TC_CFG.get("top_n_terms", 10))
TC_MIN_CONCRETE     = int(TC_CFG.get("min_concrete_core_terms", 3))
TC_GENERIC_TERMS    = set(TC_CFG.get("generic_core_terms", []) or [])
TC_ACTION           = str(TC_CFG.get("generic_cluster_action", "drop")).lower()
assert TC_ACTION in {"drop", "demand_concrete", "pass_through"}, (
    f"topic_clustering.generic_cluster_action must be drop|demand_concrete|pass_through, got {TC_ACTION!r}"
)

# 1. Build topic records and pairwise adjacency at the configured threshold ───
topic_records = [
    {"idx": i,
     "group": row[GROUPBY_FIELD],
     "topic_id": int(row["topic_id"]),
     "top_terms": list(row["top_terms"][:TC_TOP_N])}
    for i, (_, row) in enumerate(topics_df.iterrows())
]
sigs = [frozenset(t["top_terms"]) for t in topic_records]
pairwise_jaccard = {}  # (i,j) -> jaccard, for medoid tiebreak reuse
adj = defaultdict(set)
for i, j in combinations(range(len(topic_records)), 2):
    if topic_records[i]["group"] == topic_records[j]["group"]:
        continue
    inter = len(sigs[i] & sigs[j])
    if not inter:
        continue
    jac = inter / len(sigs[i] | sigs[j])
    pairwise_jaccard[(i, j)] = jac
    pairwise_jaccard[(j, i)] = jac
    if jac >= TC_JACCARD:
        adj[i].add(j); adj[j].add(i)

# 2. Connected components -> clusters ─────────────────────────────────────────
seen, clusters = set(), []
for i in range(len(topic_records)):
    if i in seen or i not in adj:
        continue
    stack, comp = [i], []
    while stack:
        n = stack.pop()
        if n in seen:
            continue
        seen.add(n); comp.append(n)
        stack.extend(adj[n] - seen)
    if len(comp) >= 2:
        clusters.append(sorted(comp))

clustered_idxs = {k for c in clusters for k in c}
singleton_idxs = [i for i in range(len(topic_records)) if i not in clustered_idxs]

# 3. Per-cluster: shared core, distinctives, medoid, project support ──────────
# Project support: join project_topic_bridge_df, sum topic_share per project
# across cluster members, dedup, name the result cluster_support_score.
ptb = project_topic_bridge_df  # from Step 4
ptb_indexed = ptb.set_index([GROUPBY_FIELD, "topic_id"]) if not ptb.empty else None

cluster_rows = []
membership_rows = []
generic_rows = []
units_rows = []

for cid, comp in enumerate(clusters):
    members = [topic_records[k] for k in comp]
    term_sets = [set(m["top_terms"]) for m in members]
    union_terms = set().union(*term_sets)
    inter_terms = set.intersection(*term_sets) if term_sets else set()
    counts = Counter(t for s in term_sets for t in s)
    # shared_core = terms present in at least half of the members (and >=2)
    threshold_count = max(2, math.ceil(len(members) / 2))
    shared_core = [t for t, c in counts.most_common() if c >= threshold_count]
    concrete_core = [t for t in shared_core if t not in TC_GENERIC_TERMS]
    is_generic = len(concrete_core) < TC_MIN_CONCRETE

    # Per-member distinctives: terms unique to this member within the cluster
    per_member_distinct = {}
    for m in members:
        others_union = set().union(
            *[set(mm["top_terms"]) for mm in members if mm["idx"] != m["idx"]]
        )
        per_member_distinct[m["idx"]] = sorted(set(m["top_terms"]) - others_union)

    # Medoid: highest mean Jaccard to other members. Tiebreak: supporting
    # project count from project_topic_bridge_df. Final fallback: topic_id.
    def _mean_jac(i):
        others = [j for j in [mm["idx"] for mm in members] if j != i]
        if not others:
            return 0.0
        return sum(pairwise_jaccard.get((i, j), 0.0) for j in others) / len(others)

    def _support_count(m):
        if ptb_indexed is None:
            return 0
        key = (m["group"], m["topic_id"])
        try:
            sub = ptb_indexed.loc[[key]]
        except KeyError:
            return 0
        return int(sub["project_id"].nunique())

    member_scores = sorted(
        members,
        key=lambda m: (-_mean_jac(m["idx"]), -_support_count(m), m["topic_id"]),
    )
    medoid = member_scores[0]

    # Cluster support: union of member project rows, summed topic_share
    if ptb_indexed is not None:
        member_keys = [(m["group"], m["topic_id"]) for m in members]
        member_keys_in_ptb = [k for k in member_keys if k in ptb_indexed.index]
        if member_keys_in_ptb:
            sup = ptb_indexed.loc[member_keys_in_ptb].reset_index()
            support_by_pid = (sup.groupby("project_id", as_index=False)
                                 .agg(cluster_support_score=("topic_share", "sum"),
                                      n_member_topics=("topic_id", "nunique")))
            support_by_pid = support_by_pid.sort_values("cluster_support_score", ascending=False)
            n_supporting = len(support_by_pid)
            top_supporters = support_by_pid["project_id"].head(50).tolist()
        else:
            support_by_pid = pd.DataFrame(columns=["project_id", "cluster_support_score", "n_member_topics"])
            n_supporting = 0
            top_supporters = []
    else:
        support_by_pid = pd.DataFrame(columns=["project_id", "cluster_support_score", "n_member_topics"])
        n_supporting = 0
        top_supporters = []

    groups_present = sorted({m["group"] for m in members})
    variation_ratio = round(
        (len(union_terms) - len(inter_terms)) / max(len(union_terms), 1), 3
    )

    cluster_row = {
        "cluster_id": cid,
        "n_topics": len(members),
        "n_groups": len(groups_present),
        "shared_core": shared_core,
        "concrete_shared_core": concrete_core,
        "n_shared": len(shared_core),
        "n_concrete_shared": len(concrete_core),
        "variation_ratio": variation_ratio,
        "is_generic": is_generic,
        "medoid_group": medoid["group"],
        "medoid_topic_id": medoid["topic_id"],
        "medoid_top_terms": medoid["top_terms"],
        "groups_present": groups_present,
        "n_supporting_projects": n_supporting,
        "top_supporting_project_ids": top_supporters,
    }
    cluster_rows.append(cluster_row)

    for m in members:
        membership_rows.append({
            "cluster_id": cid,
            GROUPBY_FIELD: m["group"],
            "topic_id": m["topic_id"],
            "is_medoid": (m["idx"] == medoid["idx"]),
            "distinctive_terms": per_member_distinct[m["idx"]],
            "top_terms": m["top_terms"],
        })

    if is_generic:
        generic_rows.append(cluster_row)
        if TC_ACTION == "drop":
            continue
        # demand_concrete or pass_through still emits a unit; downstream prompt
        # branch is responsible for handling is_generic=True appropriately.

    units_rows.append({
        "unit_id": f"cluster_{cid}",
        "unit_type": "cluster",
        "cluster_id": cid,
        "group": None,
        "topic_id": None,
        "is_generic": is_generic,
        "n_topics": len(members),
        "n_groups": len(groups_present),
        "n_supporting_projects": n_supporting,
        "shared_core": shared_core,
        "concrete_shared_core": concrete_core,
        "variation_ratio": variation_ratio,
        "medoid_top_terms": medoid["top_terms"],
        "groups_present": groups_present,
    })

# 4. Singleton units ──────────────────────────────────────────────────────────
for i in singleton_idxs:
    t = topic_records[i]
    n_sup = 0
    if ptb_indexed is not None:
        try:
            n_sup = int(ptb_indexed.loc[[(t["group"], t["topic_id"])]]["project_id"].nunique())
        except KeyError:
            n_sup = 0
    units_rows.append({
        "unit_id": f"singleton_{t['group']}_{t['topic_id']}",
        "unit_type": "singleton",
        "cluster_id": None,
        "group": t["group"],
        "topic_id": t["topic_id"],
        "is_generic": False,
        "n_topics": 1,
        "n_groups": 1,
        "n_supporting_projects": n_sup,
        "shared_core": None,
        "concrete_shared_core": None,
        "variation_ratio": None,
        "medoid_top_terms": t["top_terms"],
        "groups_present": [t["group"]],
    })

analysis_units_df = pd.DataFrame(units_rows)
cluster_membership_df = pd.DataFrame(membership_rows)
generic_clusters_df = pd.DataFrame(generic_rows)

# 5. Summary printout ─────────────────────────────────────────────────────────
n_clusters_kept = analysis_units_df.query("unit_type == 'cluster'").shape[0]
n_singletons    = analysis_units_df.query("unit_type == 'singleton'").shape[0]
n_generic_drop  = len(generic_clusters_df) if TC_ACTION == "drop" else 0

print("─" * 72)
print("TOPIC CLUSTERING LAYER")
print("─" * 72)
print(f"  enabled                        = {TC_ENABLED}")
print(f"  input topics                   = {len(topic_records)}")
print(f"  clusters kept as units         = {n_clusters_kept}")
print(f"  singletons                     = {n_singletons}")

# 6. Save artifacts ───────────────────────────────────────────────────────────
analysis_units_df.to_csv(OUT("analysis", "analysis_units.csv"), index=False)
cluster_membership_df.to_csv(OUT("analysis", "cluster_membership.csv"), index=False)
generic_clusters_df.to_csv(OUT("analysis", "generic_clusters.csv"), index=False)


────────────────────────────────────────────────────────────────────────
TOPIC CLUSTERING LAYER
────────────────────────────────────────────────────────────────────────
  enabled                        = True
  input topics                   = 135
  clusters kept as units         = 17
  singletons                     = 82


---
## Step 5 — LLM Topic Labeling

One API call per topic using compressed input — never raw essay text at scale.
Representative snippets are selected by NMF weight (highest-loading projects),
not randomly. Parallel execution via `ThreadPoolExecutor`.

Parse failures are stored with enough metadata to debug later; failed topics are
excluded from synthesis but do not halt the run.

In [11]:
# ── LLM topic labeling (forks on topic_clustering.enabled) ────────────────────
# When topic_clustering.enabled is True, dispatches per analysis_units_df row:
#   unit_type == "cluster"   → cluster prompt, _label_cluster_with_retry
#   unit_type == "singleton" → original topic prompt, _label_with_retry
# When False, runs the legacy per-topic path against topics_df unchanged.
#
# Outputs:
#   results              — all labels (cluster + singleton) in unit_order
#   unit_labels_df       — DataFrame of all successful labels with unit_type
#   labels_df            — legacy-compatible singleton-only DataFrame
#                          (also includes singletons-from-topics in legacy mode)
#   llm_topic_labels.json — full results, ordered

# Read topic_clustering config from either top-level or under analysis.
TC_CFG = (
    CFG.get("topic_clustering")
    or CFG.get("analysis", {}).get("topic_clustering")
    or {}
)
TC_ENABLED = bool(TC_CFG.get("enabled", False))

# ── Topic prompt (singleton path + legacy path) ───────────────────────────────
SYSTEM = (
    "You are an NLP analyst reviewing NMF topic clusters from DonorsChoose teacher "
    "essays. Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Input characteristics:\n"
    "- All tokens shown have been preprocessed: lowercased, lemmatized, deduplicated within "
    "each project, and filtered against a corpus-wide stopword list. Common stopwords "
    "('the', 'and', 'for', 'this', 'student', 'classroom', 'project', 'school', 'teacher', etc.) "
    "and very-rare terms have been removed before TF-IDF.\n"
    "- The 'Top NMF terms' are the highest-weighted terms NMF assigned to this topic and "
    "are the strongest single evidence of the topic's content.\n"
    "- The 'Representative project tokens' are token-level snippets from the highest-loading "
    "projects, not raw essay prose. Do not quote them as if they were sentences. Do not "
    "infer narrative or rhetorical style from token order or co-occurrence within a snippet.\n\n"
    "Token naming conventions you may see in topic terms:\n"
    "  __framing_[name]__ = rhetorical framing, tone, mechanism, or persuasion signal\n"
    "  __subject_[name]__ = subject/domain/content signal\n"
    "  __industry_[name]__ = workforce-development industry or skill-domain signal\n"
    "  __request_[name]__ = material/request-topology signal\n"
    "  __context_[name]__ = contextual school, community, attendance, safety, or access signal\n"
    "  __sensitive_context_[name]__ = direct sensitive-context signal; describe carefully and only when supported\n"
    "  __cat_[name]__ = legacy subject matter category token\n"
    "  __sub_[name]__ = legacy subcategory token\n"
    "These injected tokens are analyst-curated semantic signals. When they appear alongside compatible "
    "terms or snippets, treat them as important evidence about the topic's function, context, or framing, "
    "not as incidental noise. They may help distinguish superficially similar topics. However, do not use "
    "an injected token as standalone proof, and do not let it override direct contradictions in the concrete "
    "terms or representative snippets.\n"
    "Never output raw tokens such as __framing_*__, __subject_*__, __industry_*__, __request_*__, "
    "__context_*__, __sensitive_context_*__, __cat_*__, __sub_*__, or snake_case token names. "
    "Always translate such signals into plain English.\n\n"

    "Language constraints:\n"
    "- Do not use em dashes.\n"
    "- Do not use reveal-style phrasing such as 'not just X,' 'really about Y,' or 'what looks like X is Y.'\n"
    "- Do not use named places.\n\n"

    "Your job is to produce a stable canonical topic label and one dense grounded description.\n"
    "Do not optimize for cleverness, vividness, or donor-facing insight language. "
    "Optimize for specificity, repeatability, plain-English clarity, and preservation of useful evidence.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Prefer the most specific defensible mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Do not try to pack every nuance into proposed_label.\n"
    "- Do not collapse topics into broad umbrella labels like technology, literacy, "
    "engagement, classroom supplies, or social-emotional learning if the evidence "
    "supports a narrower interpretation.\n"
    "- Preserve concrete signals such as named programs, pedagogies, student populations, "
    "classroom routines, and rhetorical framing when clearly supported.\n\n"

    "Rules for description:\n"
    "- description is two to four plain-English sentences. Stay at or below four sentences. "
    "Fewer is fine when fewer is true; do not add a sentence to fill space.\n"
    "- The first sentence must name the dominant mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Subsequent sentences may add only evidence-supported nuance: population or student-condition "
    "signals; classroom function, setting, or routine signals; rhetorical framing or context signals; "
    "or a clearly secondary theme that coexists with the dominant theme.\n"
    "- Each sentence must be evidence-grounded, concrete, and plain English.\n"
    "- Prefer concrete natural-language phrasing over abstract wording.\n"
    "- Do not write implications, recommendations, donor-facing interpretation, or strategy commentary "
    "in any sentence.\n"
    "- Do not use raw preprocessing tokens or token-like jargon in any sentence.\n"
    "- All language constraints apply to every sentence: no em dashes, no reveal-style phrasing, "
    "and no named places.\n"
    "- The description is the primary evidence summary the downstream synthesis layer sees. "
    "Be specific and concrete rather than impressionistic.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for: redundancy with another topic in the same group, coherence problems "
    "too tangled to fit inside description, or sensitive-context signals worth flagging separately "
    "for downstream review.\n"
    "- Do not use notes as a second description. Secondary themes belong in description.\n"
    "- If coherence_flag is mixed, briefly name the colliding subthemes here.\n"
    "- If coherence_flag is redundant, briefly state the nature of the overlap.\n"
    "- If coherence_flag is unclear, briefly state why the topic is too scattered.\n"
    "- Do not restate the description in notes.\n\n"

    "coherence_flag definitions:\n"
    "  coherent   — one dominant theme, mechanism, population, or use case clearly leads, "
    "even if secondary signals are present.\n"
    "  mixed      — no single dominant theme clearly leads and two or more distinguishable "
    "subthemes are truly colliding.\n"
    "  redundant  — this topic's terms and snippets substantially duplicate another "
    "topic in the same group, not merely overlap in subject area.\n"
    "  unclear    — terms are too scattered or generic to support a defensible label.\n\n"

    "Important tie-breaker:\n"
    "- If one theme is primary and another is secondary, mark the topic coherent, not mixed, "
    "and include the secondary theme in description rather than in notes.\n"
    "- Use mixed only when no single dominant theme clearly leads.\n"
    "- If two phrasings are both plausible, choose the more literal, reusable, and plain-English one."
)

PROMPT = (
    f"Group field: {GROUPBY_FIELD} — {GROUP_DESCRIPTION}\n"
    f"Group value: {{group_value}}{{bin_line}}\n"
    "Topic {topic_id}\n"
    "Top unigrams : {unigrams}\n"
    "Top bigrams  : {bigrams}\n"
    "Top NMF terms: {nmf_terms}\n"
    "Representative project tokens:\n"
    "{snippets}\n\n"

    "Instructions:\n"
    "- proposed_label must be a short canonical noun phrase, usually 3 to 7 words. "
    "Do not pack nuance into the label.\n"
    "- description must be two to four plain-English sentences. Stay at or below four sentences. "
    "Fewer is fine when fewer is true.\n"
    "- The first sentence must name the dominant mechanism, request type, intervention, "
    "classroom routine, or use case.\n"
    "- Subsequent sentences may add only evidence-supported nuance: population or student-condition "
    "signals; classroom function, setting, or routine signals; rhetorical framing or context signals; "
    "or a clearly secondary theme that coexists with the dominant theme.\n"
    "- Prefer concrete mechanism and classroom function over broad category language.\n"
    "- Translate any framing/category/subcategory token signals into normal language; never copy raw token strings.\n"
    "- If the topic appears broad, identify the narrower mechanism, use case, or population "
    "if the evidence supports it.\n"
    "- Use coherence_flag='mixed' only when no single dominant theme clearly leads.\n"
    "- If one theme is primary and another is secondary, mark the topic coherent and include "
    "the secondary theme in description.\n"
    "- notes should usually be empty; reserve notes for redundancy, tangled coherence, or "
    "sensitive-context flagging.\n"
    "- Do not write donor-facing insights, implications, recommendations, or rhetorical flourishes.\n\n"

    "Output requirements:\n"
    "- proposed_label: short and stable, 3 to 7 words, do not pack nuance here\n"
    "- description: 2 to 4 plain-English sentences carrying the primary theme plus any clearly secondary signal\n"
    '- coherence_flag: one of "coherent", "mixed", "redundant", "unclear"\n'
    "- notes: usually empty; reserved for redundancy, tangled coherence, or sensitive-context flagging\n\n"

    f"Return a JSON object with exactly these keys:\n"
    f"{GROUPBY_FIELD}, topic_id, proposed_label, description, coherence_flag, notes"
)

# ── Cluster prompt (cluster path only) ────────────────────────────────────────
CLUSTER_SYSTEM = (
    "You are an NLP analyst labeling cross-group NMF topic clusters from DonorsChoose "
    "teacher essays. A cluster is a recurring classroom-need family: multiple NMF topics "
    "from different group slices that share a core of terms. Your job is to label the "
    "shared family and capture how it varies across the groups that contain it.\n\n"

    "Respond ONLY with a single valid JSON object. No preamble. No markdown fences.\n\n"

    "Input characteristics:\n"
    "- All tokens shown have been preprocessed: lowercased, lemmatized, deduplicated within "
    "each project, and filtered against a corpus-wide stopword list. Common stopwords "
    "('the', 'and', 'for', 'this', 'student', 'classroom', 'project', 'school', 'teacher', etc.) "
    "and very-rare terms have been removed before TF-IDF.\n"
    "- 'Shared core' lists terms present in at least half of the cluster's member topics' top "
    "terms; treat these as the cluster's identity.\n"
    "- Each member's 'Distinctive terms' are terms in only that member's top terms among the "
    "cluster. These are the strongest evidence for variation_notes.\n"
    "- The 'Representative project tokens' lines are token-level snippets from the highest-loading "
    "project per member group, not raw essay prose. Do not quote them as sentences.\n\n"

    "Token naming conventions you may see in cluster terms:\n"
    "  __framing_[name]__ = rhetorical framing or tone signal\n"
    "  __subject_[name]__ = subject/domain signal\n"
    "  __industry_[name]__ = workforce-development industry signal\n"
    "  __request_[name]__ = material/request-topology signal\n"
    "  __context_[name]__ = contextual school/community signal\n"
    "  __sensitive_context_[name]__ = direct sensitive-context signal\n"
    "  __cat_[name]__ = legacy category token\n"
    "  __sub_[name]__ = legacy subcategory token\n"
    "Translate these into plain English; never output raw token strings.\n\n"

    "Language constraints:\n"
    "- Do not use em dashes.\n"
    "- Do not use reveal-style phrasing.\n"
    "- Do not use named places.\n\n"

    "Rules for proposed_label:\n"
    "- proposed_label is a short canonical noun phrase, usually 3 to 7 words.\n"
    "- Name the recurring classroom-need family, not any single member's specific angle.\n"
    "- Prefer concrete request/mechanism language over abstract umbrella terms.\n"
    "- Do not pack every member's distinctive into the label.\n\n"

    "Rules for description:\n"
    "- description is two to four plain-English sentences. Stay at or below four sentences. "
    "Fewer is fine when fewer is true; do not add a sentence to fill space.\n"
    "- Describe what the cluster's shared core represents as a recurring need pattern.\n"
    "- Include additional structural signals only when they are genuinely shared across all or nearly all members, "
    "such as a common classroom function, student-condition signal, operating pressure, or request type.\n"
    "- Do not enumerate per-group differences in description; variation_notes is for that.\n"
    "- Do not write donor-facing insights, implications, recommendations, or strategy commentary.\n"
    "- Do not use raw preprocessing tokens or token-like jargon.\n\n"

    "Rules for variation_notes:\n"
    "- variation_notes is a list with exactly one entry per member group listed in 'Groups present'.\n"
    "- Each entry has two fields: 'group' (exact group value from Groups present) and "
    "'distinctive_angle' (plain-English description of what makes this member's contribution distinctive).\n"
    "- distinctive_angle should reference that member's distinctive terms and top NMF terms.\n"
    "- If a member has no meaningful distinctive angle beyond the shared core, set distinctive_angle "
    "to exactly 'no distinctive angle'.\n"
    "- Do not invent groups not in Groups present. Do not omit any group in Groups present. "
    "Do not list the same group more than once.\n\n"

    "Rules for coherence_flag:\n"
    "  coherent  — one shared family clearly leads, variation across groups is consistent and complementary.\n"
    "  mixed     — the shared core is real but variation notes describe genuinely incompatible angles.\n"
    "  redundant — the cluster has no concrete variation; every member says roughly the same thing.\n"
    "  unclear   — cannot tell what binds the cluster.\n\n"

    "Rules for notes:\n"
    "- notes should usually be empty.\n"
    "- Use notes only for real edge cases.\n"
    "- If coherence_flag is mixed, briefly name the colliding angles.\n"
    "- If coherence_flag is redundant, briefly say so.\n"
    "- If coherence_flag is unclear, briefly state why.\n"
    "- Do not restate description in notes."
)

CLUSTER_PROMPT = (
    f"Group field: {GROUPBY_FIELD} — {GROUP_DESCRIPTION}\n\n"
    "Cluster {cluster_id}\n"
    "Members          : {n_topics} topics across {n_groups} groups\n"
    "Shared core      : {shared_core}\n"
    "Concrete core    : {concrete_shared_core}\n"
    "Medoid group     : {medoid_group}\n"
    "Medoid top terms : {medoid_top_terms}\n"
    "Groups present   : {groups_present}\n\n"
    "Per-member detail:\n"
    "{members_block}\n\n"

    "Output requirements:\n"
    "- proposed_label : short noun phrase naming the shared family\n"
    "- description    : 2 to 4 plain-English sentences about the shared core, not per-group variation\n"
    "- variation_notes: list with one entry per group in Groups present, each appearing exactly once\n"
    "- coherence_flag : one of 'coherent', 'mixed', 'redundant', 'unclear'\n"
    "- notes          : usually empty\n\n"

    "Return a JSON object with exactly these keys: "
    "cluster_id, proposed_label, description, variation_notes, coherence_flag, notes"
)

# ── pid_text snippet lookup (dedup before set_index to handle exploded rows) ──
pid_text = (
    df.drop_duplicates("project_id")
      .set_index("project_id")["tokens"]
      .apply(lambda t: " ".join(coerce_token_list(t)[:100]))
)

# ── Dispatch ──────────────────────────────────────────────────────────────────
if TC_ENABLED:
    if "analysis_units_df" not in dir() or analysis_units_df.empty:
        raise RuntimeError(
            "topic_clustering.enabled is True but analysis_units_df is missing or empty. "
            "Re-run the topic clustering layer cell before Step 5."
        )

    cluster_units = analysis_units_df[analysis_units_df["unit_type"] == "cluster"]
    singleton_units = analysis_units_df[analysis_units_df["unit_type"] == "singleton"]
    print(f"Step 5 dispatch: {len(cluster_units)} cluster(s), {len(singleton_units)} singleton(s).")

    # Stable unit ordering so the output JSON is deterministic across runs.
    unit_order = {
        str(row["unit_id"]): i
        for i, (_, row) in enumerate(analysis_units_df.iterrows())
    }

    # Build cluster inputs.
    cluster_inputs = [
        build_cluster_input(
            row,
            membership_df=cluster_membership_df,
            weights_df=weights_df,
            pid_text=pid_text,
            groupby_field=GROUPBY_FIELD,
            n_representative_per_member=1,
            top_terms_per_member=8,
        )
        for _, row in cluster_units.iterrows()
    ]

    # Build singleton inputs by reconstructing a topic-row shape per unit.
    # Carry the precomputed unit_id from analysis_units_df so we never
    # synthesize unsafe IDs from raw group values.
    singleton_inputs = []
    for _, srow in singleton_units.iterrows():
        topic_row = topics_df[
            (topics_df[GROUPBY_FIELD] == srow["group"])
            & (topics_df["topic_id"] == srow["topic_id"])
        ]
        if topic_row.empty:
            append_warning(
                WARNINGS_PATH, "03_insights_generation", "SINGLETON_TOPIC_MISSING",
                f"Singleton unit references topic not in topics_df: "
                f"{srow['group']} / {srow['topic_id']}",
                context={"group": str(srow["group"]), "topic_id": int(srow["topic_id"])},
            )
            continue
        inp = build_input(
            topic_row.iloc[0],
            weights_df=weights_df,
            pid_text=pid_text,
            groupby_field=GROUPBY_FIELD,
            n_representative=N_REPRESENTATIVE,
            top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
        )
        inp["unit_id"] = str(srow["unit_id"])
        singleton_inputs.append(inp)

    # ── Sanity checks before paid LLM calls ──────────────────────────────────
    if cluster_inputs:
        sample_cluster = cluster_inputs[0]
        if isinstance(sample_cluster.get("shared_core"), str):
            raise TypeError(
                "Cluster shared_core is still a string after coercion. "
                "Check coerce_token_list() before running LLM calls."
            )
        if sample_cluster.get("members"):
            sample_member = sample_cluster["members"][0]
            if isinstance(sample_member.get("top_terms"), str):
                raise TypeError(
                    "Cluster member top_terms is still a string after coercion. "
                    "Check coerce_token_list() before running LLM calls."
                )

    if singleton_inputs:
        sample_singleton = singleton_inputs[0]
        if not sample_singleton.get("group_value") or sample_singleton.get("topic_id") is None:
            raise ValueError(
                "Singleton input is missing group_value or topic_id. "
                "Check singleton unit reconstruction before running LLM calls."
            )
    
    results = []
    with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
        futures = {}
        for inp in cluster_inputs:
            futures[executor.submit(
                _label_cluster_with_retry,
                inp,
                client=client,
                model_labeling=MODEL_LABELING,
                system_prompt=CLUSTER_SYSTEM,
                user_prompt_template=CLUSTER_PROMPT,
                warnings_path=WARNINGS_PATH,
                max_retries=MAX_RETRIES,
                retry_delay_seconds=LLM_RETRY_DELAY,
                service_tier=LLM_SERVICE_TIER,
                flex_attempts=LLM_FLEX_ATTEMPTS,
                fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
                timeout_seconds=LLM_TIMEOUT_SECONDS,
                flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
            )] = ("cluster", inp)
        for inp in singleton_inputs:
            futures[executor.submit(
                _label_with_retry,
                inp,
                client=client,
                model_labeling=MODEL_LABELING,
                system_prompt=SYSTEM,
                user_prompt_template=PROMPT,
                groupby_field=GROUPBY_FIELD,
                warnings_path=WARNINGS_PATH,
                max_retries=MAX_RETRIES,
                retry_delay_seconds=LLM_RETRY_DELAY,
                service_tier=LLM_SERVICE_TIER,
                flex_attempts=LLM_FLEX_ATTEMPTS,
                fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
                timeout_seconds=LLM_TIMEOUT_SECONDS,
                flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
            )] = ("singleton", inp)
        for future in as_completed(futures):
            kind, inp = futures[future]
            obj = future.result()
            if kind == "singleton":
                obj["unit_type"] = "singleton"
                obj["unit_id"] = inp["unit_id"]
            results.append(obj)
            disp = obj.get("proposed_label", "?")
            tag = (
                f"cluster {inp['cluster_id']}"
                if kind == "cluster"
                else f"{inp['group_value']} / topic {inp['topic_id']}"
            )
            print(f"  [{kind:9s}] {tag} → {disp}")

    # Sort results into stable unit order.
    results = sorted(
        results,
        key=lambda r: unit_order.get(str(r.get("unit_id", "")), 10**9),
    )

    with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False, default=str)

    # ── Build two DataFrames so downstream can choose its shape ──────────────
    # unit_labels_df: all successful labels with unit_type column.
    # labels_df:      legacy-compatible shape (singletons only, with
    #                 GROUPBY_FIELD and topic_id), so existing helpers like
    #                 build_topic_lines() keep working until they are
    #                 explicitly upgraded for cluster-aware Step 6.
    successful = [r for r in results if not r.get("parse_error")]
    unit_labels_df = pd.DataFrame(successful)

    singleton_label_rows = []
    for r in successful:
        if r.get("unit_type") != "singleton":
            continue
        # Recover GROUPBY_FIELD and topic_id from the matching singleton_unit row.
        u = singleton_units[singleton_units["unit_id"] == r.get("unit_id")]
        if u.empty:
            continue
        srow = u.iloc[0]
        legacy_row = {
            GROUPBY_FIELD: str(srow["group"]),
            "topic_id": int(srow["topic_id"]),
            "proposed_label": r.get("proposed_label"),
            "description": r.get("description"),
            "coherence_flag": r.get("coherence_flag"),
            "notes": r.get("notes"),
            "model": r.get("model"),
            "timestamp": r.get("timestamp"),
            "unit_id": r.get("unit_id"),
            "unit_type": "singleton",
        }
        singleton_label_rows.append(legacy_row)
    labels_df = pd.DataFrame(singleton_label_rows)

    n_clusters_ok = sum(1 for r in successful if r.get("unit_type") == "cluster")
    n_singletons_ok = sum(1 for r in successful if r.get("unit_type") == "singleton")
    n_errors = len(results) - len(successful)
    n_validation_warn = sum(1 for r in successful if r.get("validation_warning"))
    print(f"\n{len(results)} labels saved  "
          f"(clusters ok: {n_clusters_ok}, singletons ok: {n_singletons_ok}, "
          f"errors: {n_errors}, validation warnings: {n_validation_warn})")
    print(f"unit_labels_df: {len(unit_labels_df)} rows  |  "
          f"labels_df (legacy singleton shape): {len(labels_df)} rows")

else:
    # ── Legacy path: per-topic labeling against topics_df, unchanged ──────────
    topic_inputs = [
        build_input(
            t,
            weights_df=weights_df,
            pid_text=pid_text,
            groupby_field=GROUPBY_FIELD,
            n_representative=N_REPRESENTATIVE,
            top_terms_in_prompt=TOP_TERMS_IN_PROMPT,
        )
        for _, t in topics_df.iterrows()
    ]

    topic_order = {
        (_norm_group_value(inp["group_value"]), int(inp["topic_id"])): i
        for i, inp in enumerate(topic_inputs)
    }

    results = []
    with ThreadPoolExecutor(max_workers=LABELING_MAX_WORKERS) as executor:
        futures = {
            executor.submit(
                _label_with_retry,
                inp,
                client=client,
                model_labeling=MODEL_LABELING,
                system_prompt=SYSTEM,
                user_prompt_template=PROMPT,
                groupby_field=GROUPBY_FIELD,
                warnings_path=WARNINGS_PATH,
                max_retries=MAX_RETRIES,
                retry_delay_seconds=LLM_RETRY_DELAY,
                service_tier=LLM_SERVICE_TIER,
                flex_attempts=LLM_FLEX_ATTEMPTS,
                fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
                timeout_seconds=LLM_TIMEOUT_SECONDS,
                flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
            ): inp
            for inp in topic_inputs
        }
        for future in as_completed(futures):
            inp = futures[future]
            obj = future.result()
            results.append(obj)
            print(f"  {inp['group_value']} / topic {inp['topic_id']} → {obj.get('proposed_label', '?')}")

    bad_sort_keys = []
    for r in results:
        norm_key = (
            _norm_group_value(r.get(GROUPBY_FIELD, "")),
            _safe_topic_id(r.get("topic_id", -1)),
        )
        if norm_key not in topic_order:
            bad_sort_keys.append({"group": r.get(GROUPBY_FIELD, ""), "topic_id": r.get("topic_id", "")})

    if bad_sort_keys:
        raise ValueError(
            f"LABELING_SORT_KEY_MISMATCH: {len(bad_sort_keys)} label result(s) did not match the input topic order. "
            f"Examples: {bad_sort_keys[:5]}"
        )

    results = sorted(
        results,
        key=lambda r: topic_order[
            (_norm_group_value(r.get(GROUPBY_FIELD, "")), _safe_topic_id(r.get("topic_id", -1)))
        ],
    )

    with open(OUT("analysis", "llm_topic_labels.json"), "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False, default=str)

    print(f"\n{len(results)} labels saved")

    # In legacy mode, labels_df is built in the post-processing cell as before.
    # unit_labels_df is not defined; downstream legacy-mode code should not
    # reference it.

Step 5 dispatch: 17 cluster(s), 82 singleton(s).
  [singleton] framing_experiential_hands_on_learning / topic 0 → interactive digital learning tools
  [cluster  ] cluster 7 → Reading interest books
  [cluster  ] cluster 0 → Science inquiry exploration
  [cluster  ] cluster 16 → English support for language learners
  [cluster  ] cluster 9 → Flexible classroom seating
  [cluster  ] cluster 12 → self-regulation calming spaces
  [cluster  ] cluster 14 → Integrated STEM learning
  [cluster  ] cluster 10 → Science lab data collection
  [cluster  ] cluster 5 → Organized basic science supplies
  [cluster  ] cluster 13 → Hands-on engineering exploration
  [cluster  ] cluster 11 → Dry erase classroom supplies
  [cluster  ] cluster 8 → Fine motor play materials
  [singleton] framing_experiential_hands_on_learning / topic 2 → Hands-On Social Emotional Learning
  [singleton] framing_experiential_hands_on_learning / topic 4 → hands-on therapy tools
  [cluster  ] cluster 3 → Hands-on learning suppor

In [12]:
# ── Label post-processing ──────────────────────────────────────────────────
# Keep only parseable labeling results and validate the legacy topic-label shape.
#
# In topic_clustering mode:
#   results        = mixed cluster + singleton labels
#   unit_labels_df = all successful unit labels
#   labels_df      = singleton-only, legacy-compatible topic labels
#
# Do NOT rebuild labels_df from all results in clustering mode, because cluster
# rows intentionally do not have GROUPBY_FIELD/topic_id and will appear as NaN.

parse_errors = [r for r in results if r.get("parse_error")]
if parse_errors:
    print(f"WARNING: {len(parse_errors)} labels failed JSON parse — excluded from synthesis:")
    for e in parse_errors:
        if e.get("unit_type") == "cluster":
            print(f"  cluster {e.get('cluster_id', '?')}")
        else:
            print(f"  {e.get(GROUPBY_FIELD, '?')} / topic {e.get('topic_id', '?')}")

TC_CFG = (
    CFG.get("topic_clustering")
    or CFG.get("analysis", {}).get("topic_clustering")
    or {}
)
TC_ENABLED = bool(TC_CFG.get("enabled", False))

if TC_ENABLED:
    # Step 5 already created:
    #   unit_labels_df = all successful cluster + singleton labels
    #   labels_df      = singleton-only legacy-compatible labels
    if "unit_labels_df" not in globals():
        raise RuntimeError(
            "topic_clustering.enabled is True but unit_labels_df is missing. "
            "Re-run Step 5."
        )
    if "labels_df" not in globals():
        raise RuntimeError(
            "topic_clustering.enabled is True but labels_df is missing. "
            "Step 5 should create singleton-only labels_df."
        )

    # Defensive copy so later cells can mutate safely.
    unit_labels_df = unit_labels_df.copy()
    labels_df = labels_df.copy()

else:
    # Legacy mode: all successful results are ordinary topic labels.
    labels_df = pd.DataFrame([r for r in results if not r.get("parse_error")])
    unit_labels_df = pd.DataFrame()

assert GROUPBY_FIELD in labels_df.columns, (
    f"labels_df missing '{GROUPBY_FIELD}' — re-run labeling for this groupby field"
)
assert "topic_id" in labels_df.columns, (
    "labels_df missing 'topic_id' — re-run labeling or check singleton label construction"
)

# Drop rows that are not valid legacy topic labels. This should be a no-op in
# both legacy mode and corrected clustering mode, but protects against partial
# reruns where cluster rows leaked into labels_df.
labels_df = labels_df.dropna(subset=[GROUPBY_FIELD, "topic_id"]).copy()

# Source-of-truth allowed groups come from the source topic table, not model output.
ALLOWED_GROUP_VALUES = [
    str(g)
    for g in topics_df[GROUPBY_FIELD].dropna().drop_duplicates().tolist()
]

invalid_groups = sorted(
    set(labels_df[GROUPBY_FIELD].astype(str).unique()) - set(ALLOWED_GROUP_VALUES)
)
if invalid_groups:
    raise ValueError(
        f"Invalid {GROUPBY_FIELD} values returned during labeling: {invalid_groups}"
    )

# Normalize types after validation so later joins are stable.
labels_df[GROUPBY_FIELD] = labels_df[GROUPBY_FIELD].astype(str)
labels_df["topic_id"] = labels_df["topic_id"].astype(int)

# Compute supporting-project counts at the analysis-unit level (topic OR
# cluster) and a run-median multiplier so synthesis can weigh strong vs thin
# units. Multiplier is normalized against the median support count across
# ALL units (clusters + singletons combined) so the value means the same
# thing on every line: "this unit is supported by Nx as many distinct
# projects as the median analysis unit in this run."
if "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty:
    # Step 1: per-topic supporting-project count.
    _per_topic_support = (
        project_topic_bridge_df
        .groupby([GROUPBY_FIELD, "topic_id"], observed=True)["project_id"]
        .nunique()
        .reset_index()
        .rename(columns={"project_id": "support_count"})
    )
    _per_topic_support["topic_id"] = _per_topic_support["topic_id"].astype(int)
    _per_topic_support[GROUPBY_FIELD] = _per_topic_support[GROUPBY_FIELD].astype(str)

    # Step 2: per-cluster supporting-project count (deduped across members).
    # Built from cluster_membership_df + project_topic_bridge_df so the count
    # is unique projects across all member topics.
    _per_cluster_support = pd.DataFrame(columns=["cluster_id", "support_count"])
    if (TC_ENABLED
            and "cluster_membership_df" in dir()
            and not cluster_membership_df.empty):
        _mem = cluster_membership_df[["cluster_id", GROUPBY_FIELD, "topic_id"]].copy()
        _mem[GROUPBY_FIELD] = _mem[GROUPBY_FIELD].astype(str)
        _mem["topic_id"] = _mem["topic_id"].astype(int)
        _ptb = project_topic_bridge_df[
            [GROUPBY_FIELD, "topic_id", "project_id"]
        ].copy()
        _ptb[GROUPBY_FIELD] = _ptb[GROUPBY_FIELD].astype(str)
        _ptb["topic_id"] = _ptb["topic_id"].astype(int)
        _joined = _mem.merge(_ptb, on=[GROUPBY_FIELD, "topic_id"], how="left")
        _per_cluster_support = (
            _joined.dropna(subset=["project_id"])
            .groupby("cluster_id")["project_id"]
            .nunique()
            .reset_index()
            .rename(columns={"project_id": "support_count"})
        )
        _per_cluster_support["cluster_id"] = _per_cluster_support["cluster_id"].astype(int)

    # Step 3: shared denominator -- median across ALL unit support_counts.
    # In TC mode, this includes singletons (from _per_topic_support filtered
    # to singletons) and clusters. In legacy mode, only topic counts.
    if TC_ENABLED and "unit_labels_df" in dir() and not unit_labels_df.empty:
        # Singleton support: lookup per-topic count for singleton rows.
        _singleton_keys = unit_labels_df[
            unit_labels_df["unit_type"] == "singleton"
        ][[groupby_field if (groupby_field := GROUPBY_FIELD) else GROUPBY_FIELD, "topic_id"]].copy() \
            if False else unit_labels_df.loc[
                unit_labels_df["unit_type"] == "singleton",
                [GROUPBY_FIELD, "topic_id"]
            ].copy()
        _singleton_keys[GROUPBY_FIELD] = _singleton_keys[GROUPBY_FIELD].astype(str)
        _singleton_keys["topic_id"] = _singleton_keys["topic_id"].astype(int)
        _singleton_support = _singleton_keys.merge(
            _per_topic_support, on=[GROUPBY_FIELD, "topic_id"], how="left"
        )["support_count"].fillna(0).astype(int).tolist()
        _cluster_support_list = _per_cluster_support["support_count"].astype(int).tolist()
        _all_unit_supports = [s for s in (_singleton_support + _cluster_support_list) if s > 0]
    else:
        _all_unit_supports = [
            s for s in _per_topic_support["support_count"].astype(int).tolist() if s > 0
        ]

    _median_support = pd.Series(_all_unit_supports).median() if _all_unit_supports else 0
    if pd.notna(_median_support) and _median_support > 0:
        # Merge support_count into labels_df (covers legacy-mode topics AND
        # TC-mode singletons since labels_df is singleton-only in TC mode).
        labels_df = labels_df.merge(
            _per_topic_support, on=[GROUPBY_FIELD, "topic_id"], how="left"
        )
        labels_df["support_count"] = labels_df["support_count"].fillna(0).astype(int)
        labels_df["support_multiplier"] = (
            labels_df["support_count"] / _median_support
        ).round(2)

        # In TC mode, also merge cluster + singleton support into unit_labels_df.
        if TC_ENABLED and "unit_labels_df" in dir() and not unit_labels_df.empty:
            # Cluster rows: merge by cluster_id.
            unit_labels_df = unit_labels_df.merge(
                _per_cluster_support, on="cluster_id", how="left"
            )
            # Singleton rows: merge per-topic support by (group, topic_id).
            # unit_labels_df cluster rows have NaN for GROUPBY_FIELD/topic_id;
            # the merge will only fill singleton rows.
            _singleton_lookup = _per_topic_support.rename(
                columns={"support_count": "_topic_support"}
            )
            if GROUPBY_FIELD in unit_labels_df.columns and "topic_id" in unit_labels_df.columns:
                unit_labels_df[GROUPBY_FIELD] = unit_labels_df[GROUPBY_FIELD].astype("string")
                unit_labels_df["topic_id"] = pd.to_numeric(
                    unit_labels_df["topic_id"], errors="coerce"
                ).astype("Int64")
                unit_labels_df = unit_labels_df.merge(
                    _singleton_lookup,
                    left_on=[GROUPBY_FIELD, "topic_id"],
                    right_on=[GROUPBY_FIELD, "topic_id"],
                    how="left",
                )
                # Fill the unified support_count from whichever side applies.
                unit_labels_df["support_count"] = unit_labels_df["support_count"].fillna(
                    unit_labels_df["_topic_support"]
                ).fillna(0).astype(int)
                unit_labels_df = unit_labels_df.drop(columns=["_topic_support"])
            else:
                unit_labels_df["support_count"] = unit_labels_df["support_count"].fillna(0).astype(int)

            unit_labels_df["support_multiplier"] = (
                unit_labels_df["support_count"] / _median_support
            ).round(2)

        print(f"Support multiplier added: median support = {int(_median_support)} projects "
              f"(denominator: all analysis units), "
              f"labels_df range = {labels_df['support_multiplier'].min():.2f}x to "
              f"{labels_df['support_multiplier'].max():.2f}x.")
        if TC_ENABLED and "unit_labels_df" in dir() and not unit_labels_df.empty:
            _cl = unit_labels_df[unit_labels_df["unit_type"] == "cluster"]
            if not _cl.empty:
                print(f"  cluster support_multiplier range: "
                      f"{_cl['support_multiplier'].min():.2f}x to "
                      f"{_cl['support_multiplier'].max():.2f}x  "
                      f"(median: {_cl['support_multiplier'].median():.2f}x).")
    else:
        labels_df["support_count"] = 0
        labels_df["support_multiplier"] = 0.0
        if TC_ENABLED and "unit_labels_df" in dir() and not unit_labels_df.empty:
            unit_labels_df["support_count"] = 0
            unit_labels_df["support_multiplier"] = 0.0
else:
    labels_df["support_count"] = 0
    labels_df["support_multiplier"] = 0.0
    if TC_ENABLED and "unit_labels_df" in dir() and not unit_labels_df.empty:
        unit_labels_df["support_count"] = 0
        unit_labels_df["support_multiplier"] = 0.0
    append_warning(
        WARNINGS_PATH, "03_insights_generation", "LABELS_SUPPORT_UNAVAILABLE",
        "project_topic_bridge_df missing or empty; support_multiplier disabled. "
        "Synthesis will not see per-unit strength signals.",
        context={},
    )

labeled_groups = set(labels_df[GROUPBY_FIELD].unique()) if not labels_df.empty else set()
LABELING_FAILED_GROUPS = sorted(set(ALLOWED_GROUP_VALUES) - labeled_groups)

print(f"labels_df topic-label rows: {len(labels_df):,}")
if TC_ENABLED:
    print(f"unit_labels_df unit-label rows: {len(unit_labels_df):,}")

display(labels_df.groupby("coherence_flag").size().rename("count").to_frame())
labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "coherence_flag", "description"]]

Support multiplier added: median support = 825 projects (denominator: all analysis units), labels_df range = 0.02x to 2.46x.
  cluster support_multiplier range: 0.10x to 5.40x  (median: 2.35x).
labels_df topic-label rows: 82
unit_labels_df unit-label rows: 99


,count
coherence_flag,
coherent,82


,strategic_injected_tag,topic_id,proposed_label,coherence_flag,description
0,framing_experiential_hands_on_learning,0,interactive digital learning tools,coherent,Interactive digital learning tools such as lap...
1,framing_experiential_hands_on_learning,2,Hands-On Social Emotional Learning,coherent,Hands-on social emotional learning materials s...
2,framing_experiential_hands_on_learning,4,hands-on therapy tools,coherent,"Hands-on therapy tools for fine motor, visual ..."
3,framing_experiential_hands_on_learning,5,hands-on garden learning,coherent,Hands-on garden-based learning that uses plant...
4,framing_experiential_hands_on_learning,6,hands-on STEM engineering projects,coherent,Hands-on STEM engineering projects center on b...
...,...,...,...,...,...
77,subject_science_inquiry_lab_learning,7,Molecular model kits,coherent,Hands-on molecular model kits for chemistry in...
78,subject_science_inquiry_lab_learning,9,Owl Pellet Dissection Lab,coherent,Hands-on owl pellet dissection for food chain ...
79,subject_science_inquiry_lab_learning,10,science lab data analysis,coherent,This topic is about hands-on science lab inves...
80,subject_science_inquiry_lab_learning,11,Scientific calculator access,coherent,Scientific calculator access for math and scie...


---
## Step 6 — Synthesis

Produces two synthesis passes in sequence:

1. **Cross-group synthesis** — a single LLM call over all topic lines to identify
   patterns, contrasts, framing logic, and notable boundaries across the corpus.
2. **Per-group synthesis** — one LLM call per group, run in parallel, to surface
   each group's dominant internal mechanisms.

Both passes produce plain text saved to `analysis/llm_synthesis_*.txt` and are
assembled in Step 7 as the evidence base for the structured JSON insight call.

In [13]:
# ── SYNTHESIS — cross-group + per-group loops ─────────────────────────────
# First synthesize the whole landscape, then synthesize each group in parallel.

SYNTHESIS_SYSTEM = '''You are a senior program analyst at an educational nonprofit.
Your job is to synthesize topic-level or unit-level evidence into stable, decision-useful analytic findings for internal strategy work.
You are not writing polished external copy and you are not doing creative interpretation.
You are performing disciplined evidence grouping.

Core objective:
- Produce findings that are specific, well-grounded, and tightly tied to the supplied evidence lines.
- Identify useful classroom, student, operational, funding, or context patterns without forcing surprise or contrast.
- Literal evidence discipline is more important than elegance.
- Do not invent recency, current-events framing, causality, prevalence, or geographic specificity that the evidence lines do not support.

Evidence-line types:
- A topic line represents one labeled NMF topic for one group.
- A cluster line, when present, represents a synthesis aid that groups related topics across one or more groups.
- Cluster lines may contain shared_core, variation, span, support, and member_topics.
- Cluster IDs are not valid citations.
- Use cluster labels, shared_core, and variation notes to understand the pattern, but cite only the member_topics listed on the cluster line.
- A singleton line behaves like a normal topic line and may be cited directly as <group_value>|<topic_id>.
- A topic line's description may contain multiple sentences naming primary and secondary evidence signals. Treat the description as the most direct statement of what the topic represents.
- Use top_terms to corroborate the description, check specificity, and catch possible overreach. Do not use top_terms to override a well-grounded description unless the description clearly conflicts with the evidence line.

Citation contract:
- Every finding must include a Supporting topics line.
- Each supporting topic must be written exactly as <group_value>|<topic_id>.
- This pipe-string format is the only canonical citation format.
- group_value must be copied verbatim from the evidence lines or member_topics list.
- Do not emit cluster IDs, topic labels, prose descriptions, dictionaries, or bare topic numbers as citations.
- If a cluster supports a finding, unfold it to the member topic citations that directly support that finding.
- Do not cite every member of a cluster automatically. Cite only the member topics that support the specific finding.

Support-weight rules:
- Evidence lines may include a `support: 2.4x` value.
- The user prompt's Support calibration block defines the denominator and thresholds.
- Use that block exactly. Do not assume a different denominator or threshold.
- Absolute supporting-project counts and group sizes, when supplied, should temper the multiplier: a high multiplier with a small absolute count is still narrow.
- Strong support increases confidence, but support size does not override direct relevance, coherence, or cross-group consistency.
- Do not promote a thin-support topic or member topic alone to a main cross-group finding unless multiple thin-support topics together describe the same pattern.
- Do not write findings that would not survive if weakly related or thin-support topics were removed.
- When two evidence lines offer similar evidence, prefer the higher-support and more coherent one as the primary citation and treat the lower-support one as corroborating.
- If support values are absent, weigh evidence lines by relevance, coherence, specificity, and breadth.

Coherence-flag rules:
- coherent means the topic or unit has one clear dominant theme and can be used directly if relevant.
- mixed means multiple themes are present; use only the dominant theme unless the collision itself is the finding.
- redundant means the topic or unit overlaps strongly with nearby evidence; avoid double-counting it.
- unclear means the topic or unit is weak, vague, or hard to interpret; use it only as secondary support if stronger evidence points the same way.

Substrate rules:
- Food, hygiene, clothing, seating, storage, sensory tools, calm spaces, headphones, basic supplies, and generic devices are common classroom substrate.
- Do not promote common substrate as a finding unless the supplied evidence shows a distinctive mechanism, constraint, classroom function, student condition, or operating pressure.
- If substrate appears only as generic classroom operating support, treat it as background, evidence, or scope rather than as a standalone finding.

Merge rules:
- Merge evidence only when it shares the same dominant mechanism, classroom function, student condition, or school-day operating pressure.
- Repeated vocabulary is not enough.
- When deciding whether two topics share the same dominant mechanism, prefer the language in their descriptions over inference from overlapping top_terms.
- Two topics whose descriptions name different mechanisms are not the same finding, even if their top_terms overlap.
- Do not merge evidence just because it involves the same product category, school setting, or broad theme.
- Keep access/accommodation, regulation/behavior, maintenance/operations, identity/belonging, safety/security, mental health context, and routine/workflow separate unless the evidence clearly shows they are the same pattern.
- If one candidate finding is broader and another is a more specific evidence-grounded version, prefer the more specific one.

Scope rules:
- Scope should name the most important boundary on the finding, not every possible caveat or segment fact.
- Do not stack centrality, grade concentration, category concentration, and non-generalization warnings in the same Scope line; choose the boundary that most changes interpretation.
- Mark sparse, single-group, sensitive-context, or narrow findings as narrow or emerging.
- Do not use broad language such as "teachers are" or "classrooms need" when the evidence is concentrated in a small direct-signal slice.
- Do not use trend language such as "growing," "rising," "increasing," or "more often" unless the supplied evidence includes an explicit time comparison.
- If groups are tag-defined slices, do not imply they are mutually exclusive segments.

Writing rules:
- Be precise, concrete, and analytical.
- Direct evidence should name concrete materials, classroom routines, request patterns, student conditions, operating contexts, or repeated classroom uses.
- Direct evidence should describe what is present in classrooms or requests, not the analytical process used to find it. Do not use topics, clusters, evidence lines, or synthesis as the grammatical subject of Direct evidence. Phrases such as "topics describe," "evidence lines show," "multiple topics converge," and "the strongest topics" are not acceptable in Direct evidence.
- Direct evidence and Interpretation must do distinct jobs. Direct evidence anchors the finding in concrete classroom facts; Interpretation explains what those facts mean. Do not let Direct evidence become a second Interpretation.
- Do not mention NMF, model, pipeline, or analytical machinery.
- You may use the word "cluster" only if the input itself uses it, but do not write about clusters in the final finding text.
- Do not write donor-facing rhetoric.
- Do not force a "looks like X, but is really Y" structure.
- Do not manufacture a reader mistake or surface misread.
- Do not use em dashes.
- Do not use named places.
- Do not invent a cleaner pattern than the evidence supports.'''.strip()

topic_lines = build_topic_lines(
    labels_df,
    GROUPBY_FIELD,
    top_terms_count=SYNTHESIS_TOP_TERMS_COUNT,
)

SYNTHESIS_PROMPT_CROSS_GROUP = '''Below is a list of evidence lines discovered from teacher project request essays on DonorsChoose, grouped by "{GROUPBY_FIELD}" ({GROUP_DESCRIPTION}).

Run context:
{RUN_CONTEXT_BLOCK}

Strategic/context caveats:
{STRATEGIC_CONTEXT_BLOCK}

Support calibration:
{SUPPORT_CALIBRATION_BLOCK}

Citation examples from this run:
{CANONICAL_SOURCE_TOPIC_EXAMPLES}

Each evidence line may contain:
- group value
- topic number or cluster number
- label
- coherence flag
- support multiplier and supporting-project count
- top terms, shared terms, or shared_core
- description, usually two to four sentences
- for clusters, member_topics in canonical <group_value>|<topic_id> format

Evidence lines:
{EVIDENCE_LINES}

Your task:
Synthesize the evidence lines into a stable cross-group analysis that will later be used to generate evidence-grounded external-facing insights.
Preserve nuance, but be disciplined about grouping.

Follow this exact workflow:
1. Review the evidence lines in the order given.
2. Identify candidate cross-group findings only when multiple topics or cluster member topics share the same dominant mechanism, classroom function, student condition, or school-day operating pressure.
3. Use cluster lines, when present, to understand shared_core and variation, but cite only member_topics.
4. Keep distinct mechanisms separate even when they live in similar product categories.
5. Rank findings by:
   a. specificity of the shared mechanism
   b. clarity of distinction from nearby themes
   c. evidence strength in labels, terms, descriptions, support counts, and coherence
   d. breadth across groups, adjusted for group size and overlap caveats
6. Treat common classroom substrate as background unless the evidence shows a distinctive cross-group mechanism.
7. Omit weak or borderline patterns rather than padding.
8. A narrower but more distinctive pattern is preferred over a broader but generic one, provided both are well-supported.

Return plain text only, using exactly this structure and these section headings:

CROSS-GROUP FINDINGS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

IMPORTANT DISTINCTIONS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

Count guidance:
- CROSS-GROUP FINDINGS: return 0 to {CROSS_MAX_INSIGHTS}. The minimum {CROSS_MIN_INSIGHTS} is permitted, not expected. Returning fewer, including zero, is the correct choice when the evidence does not clearly support more strong, non-overlapping cross-group findings.
- IMPORTANT DISTINCTIONS: return 0 to 4. Include only distinctions that are meaningfully different from the cross-group findings. An empty section is valid.
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 3. Include only boundaries that prevent overgeneralization, clarify a narrow/sparse signal, or protect against likely misinterpretation. An empty section is valid.
- For every section, fewer findings, including zero, is preferred over padded, generic, overlapping, or weak findings.
- You are rewarded for restraint. Do not create a finding merely because a minimum count exists.
- Do not split one mechanism across multiple findings to increase count.

Hard requirements:
- Every finding must include a Supporting topics line.
- Supporting topics must use exact <group_value>|<topic_id> pipe-string format.
- Never cite cluster IDs.
- Do not use bullet points.
- Do not skip section headings.
- Do not repeat the same pattern in multiple sections.
- Do not force contrast language when the evidence supports a plain pattern.
- Prefer mechanism-level explanations over broad summaries like access, engagement, or support.
- If a pattern is supported by only one group, it does not belong in CROSS-GROUP FINDINGS.
- If a distinction is real but narrow, put it in IMPORTANT DISTINCTIONS rather than inflating it into a cross-group signature.
- If a signal is sparse, sensitive, tag-specific, or narrow, say so in Scope.
- Do not use trend language unless explicit time-comparison evidence is supplied.
- Do not use named places.
- Do not use em dashes.

Write like a rigorous internal analyst, not like a speechwriter.'''.format(
    GROUPBY_FIELD=GROUPBY_FIELD,
    GROUP_DESCRIPTION=GROUP_DESCRIPTION,
    RUN_CONTEXT_BLOCK=(
        f"- Grouping field: {GROUPBY_FIELD}\n"
        f"- Group description: {GROUP_DESCRIPTION}\n"
        f"- Total topic lines in this prompt: {len(labels_df)}"
    ),
    STRATEGIC_CONTEXT_BLOCK=(
        f"- Strategic loop enabled: {STRATEGIC_LOOP_ENABLED}\n"
        f"- Strategic run metadata: {json.dumps({k: STRATEGIC_RUN_META.get(k) for k in ['strategic_area_id', 'strategic_area_label', 'split_id', 'groupby_fields', 'is_strategic_injected_tag']}, ensure_ascii=False)}"
    ),
    SUPPORT_CALIBRATION_BLOCK=(
        "Support multipliers in this prompt use the support values already attached to topic lines. "
        "Treat the supplied multiplier as the run-level topic-strength signal. "
        "Use absolute project counts only when they are present in the evidence lines."
    ),
    CANONICAL_SOURCE_TOPIC_EXAMPLES="\n".join(
        f"- {row[GROUPBY_FIELD]}|{int(row['topic_id'])}"
        for _, row in labels_df[[GROUPBY_FIELD, "topic_id"]].drop_duplicates().head(3).iterrows()
    ),
    EVIDENCE_LINES=topic_lines,
    CROSS_MIN_INSIGHTS=CROSS_MIN_INSIGHTS,
    CROSS_MAX_INSIGHTS=CROSS_MAX_INSIGHTS,
).strip()

PER_GROUP_INSTRUCTIONS = '''Your task:
Identify the strongest, most decision-useful patterns within this single group.
Do not merely restate cross-group patterns already listed above unless this group adds a distinctive mechanism, boundary, exception, or concrete evidence angle.

Follow this exact workflow:
1. Review the evidence lines in the order given.
2. Identify the dominant subthemes in the group.
3. Separate nearby themes unless they clearly share the same dominant mechanism, classroom function, student condition, or operating pressure.
4. Prefer narrower evidence-grounded findings over broad group-label summaries.
5. Use support multipliers, absolute support counts, coherence flags, and topic descriptions to distinguish central patterns from narrow ones.
6. Treat common classroom substrate as background unless the group evidence shows a distinctive mechanism.
7. Omit weak or redundant findings rather than padding.

Return plain text only, using exactly this structure and these section headings:

CORE GROUP FINDINGS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: <group_value>|<topic_id>; <group_value>|<topic_id>

Finding 2
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

IMPORTANT INTERNAL DISTINCTIONS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

BOUNDARIES OR NON-CENTRAL SIGNALS
Finding 1
Title: ...
Direct evidence: ...
Interpretation: ...
Scope: ...
Supporting topics: ...

Count guidance:
- CORE GROUP FINDINGS: return 0 to {PER_GROUP_MAX_INSIGHTS}. The minimum {PER_GROUP_MIN_INSIGHTS} is permitted, not expected. An empty section is the correct output when the group has no defensible pattern beyond what the group label already implies.
- IMPORTANT INTERNAL DISTINCTIONS: return 0 to 3. Include only distinctions that are meaningfully different from the core group findings. An empty section is valid.
- BOUNDARIES OR NON-CENTRAL SIGNALS: return 0 to 2. Include only boundaries that prevent overgeneralization, clarify a narrow/sparse signal, or protect against likely misinterpretation. An empty section is valid.
- For every section, fewer findings, including zero, is preferred over padded, generic, overlapping, or weak findings.
- You are rewarded for restraint. Do not create a finding merely because a minimum count exists.
- Do not split one mechanism across multiple findings to increase count.
- A generic group summary is worse than no group finding.

Hard requirements:
- Every finding must include a Supporting topics line.
- Supporting topics must use exact <group_value>|<topic_id> pipe-string format.
- Never cite cluster IDs.
- Do not use bullet points.
- Do not skip section headings.
- Do not repeat the same idea across multiple sections.
- Use mixed topics carefully; do not let one mixed topic dominate a whole finding unless the collision itself is the finding.
- If one theme is primary and another is secondary, keep the finding centered on the primary theme.
- Do not give generic summaries of the group label.
- Do not force a reveal, contrast, or looks-like-X-but-is-Y structure.
- Do not use trend language unless explicit time-comparison evidence is supplied.
- Do not use named places.
- Do not use em dashes.
- If the signal is sparse, sensitive, tag-specific, or narrow, say so in Scope.

Write like a rigorous internal analyst, not like a speechwriter.'''.format(
    PER_GROUP_MIN_INSIGHTS=PER_GROUP_MIN_INSIGHTS,
    PER_GROUP_MAX_INSIGHTS=PER_GROUP_MAX_INSIGHTS,
).strip()

try:
    synthesis_cross = _call_with_retry(
        SYNTHESIS_PROMPT_CROSS_GROUP,
        client=client,
        model_name=MODEL_SYNTHESIS,
        system_prompt=SYNTHESIS_SYSTEM,
        max_retries=MAX_RETRIES,
        retry_delay_seconds=LLM_RETRY_DELAY,
        service_tier=LLM_SERVICE_TIER,
        flex_attempts=LLM_FLEX_ATTEMPTS,
        fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
        timeout_seconds=LLM_TIMEOUT_SECONDS,
        flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
    )
except Exception as e:
    synthesis_cross = None
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_CROSS_GROUP_FAILED",
        "Cross-group synthesis failed after all retries",
        severity="error",
        context={
            "error": str(e),
            "max_retries": MAX_RETRIES,
            "service_tier": LLM_SERVICE_TIER,
            "flex_attempts": LLM_FLEX_ATTEMPTS,
            "fallback_service_tier": LLM_FALLBACK_SERVICE_TIER,
            "timeout_seconds": LLM_TIMEOUT_SECONDS,
        },
    )

if synthesis_cross is None:
    write_json(
        OUT("insights", "run_failed_synthesis_cross_group.json"),
        {
            "status": "failed",
            "stage": "synthesis_cross_group",
            "message": "Cross-group synthesis failed after all retries; this run cannot produce final insights.",
            "warnings_path": str(WARNINGS_PATH),
            "llm": {
                "max_retries": MAX_RETRIES,
                "service_tier": LLM_SERVICE_TIER,
                "flex_attempts": LLM_FLEX_ATTEMPTS,
                "fallback_service_tier": LLM_FALLBACK_SERVICE_TIER,
                "timeout_seconds": LLM_TIMEOUT_SECONDS,
            },
        },
    )
    raise RuntimeError(
        "Cross-group synthesis failed after all retries. "
        f"This combo is marked failed; the sweep runner will continue to the next combo. "
        f"See warnings at {WARNINGS_PATH}."
    )

with open(OUT("analysis", "llm_synthesis_cross_group.txt"), "w", encoding="utf-8") as f:
    f.write(synthesis_cross)

# Excluded groups were removed from df in the Parameters cell; they cannot
# appear in ALLOWED_GROUP_VALUES. Filter only for label coverage.
groups = [
    g for g in ALLOWED_GROUP_VALUES
    if g in set(labels_df[GROUPBY_FIELD].unique())
]

per_group_results = {}
SYNTHESIS_FAILED_GROUPS = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            synthesize_one_group,
            g,
            labels_df=labels_df,
            groupby_field=GROUPBY_FIELD,
            group_description=GROUP_DESCRIPTION,
            per_group_instructions=PER_GROUP_INSTRUCTIONS,
            client=client,
            model_name=MODEL_SYNTHESIS,
            system_prompt=SYNTHESIS_SYSTEM,
            warnings_path=WARNINGS_PATH,
            outpath_func=OUT,
            synthesis_top_terms_count=SYNTHESIS_TOP_TERMS_COUNT,
            max_retries=MAX_RETRIES,
            retry_delay_seconds=LLM_RETRY_DELAY,
            service_tier=LLM_SERVICE_TIER,
            flex_attempts=LLM_FLEX_ATTEMPTS,
            fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
            timeout_seconds=LLM_TIMEOUT_SECONDS,
            flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
        ): g
        for g in groups
    }
    for future in as_completed(futures):
        submitted_group = futures[future]
        try:
            group, result = future.result()
        except Exception as e:
            SYNTHESIS_FAILED_GROUPS.append(str(submitted_group))
            append_warning(
                WARNINGS_PATH,
                "03_insights_generation",
                "SYNTHESIS_GROUP_FAILED",
                f"Synthesis failed for group '{submitted_group}'",
                context={"group": submitted_group, "error": str(e)},
            )
            print(f"FAILED: {submitted_group} | error: {e}")
            continue

        if result is not None:
            per_group_results[group] = result
            print(f"Done: {group}")
        else:
            SYNTHESIS_FAILED_GROUPS.append(str(group))
            print(f"FAILED: {group}")

# Preserve the original group order before handing results downstream.
per_group_results = {g: per_group_results[g] for g in groups if g in per_group_results}

if not per_group_results:
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        "SYNTHESIS_ALL_GROUPS_FAILED",
        "No per-group synthesis results were produced after all retries",
        severity="error",
        context={"groups_attempted": groups, "failed_groups": SYNTHESIS_FAILED_GROUPS},
    )
    write_json(
        OUT("insights", "run_failed_synthesis_per_group.json"),
        {
            "status": "failed",
            "stage": "synthesis_per_group",
            "message": "No per-group synthesis results were produced after all retries; this run cannot produce final insights.",
            "groups_attempted": groups,
            "failed_groups": SYNTHESIS_FAILED_GROUPS,
            "warnings_path": str(WARNINGS_PATH),
        },
    )
    raise RuntimeError(
        "No per-group synthesis results were produced after all retries. "
        f"This combo is marked failed; the sweep runner will continue to the next combo. "
        f"See warnings at {WARNINGS_PATH}."
    )


Done: subject_math_manipulatives_quantitative_learning
Done: framing_problem_solving_adaptive_thinking
Done: framing_inquiry_exploration
Done: subject_science_inquiry_lab_learning
Done: subject_robotics_makerspace_3d_printing
Done: subject_engineering_design_maker_learning
Done: framing_experiential_hands_on_learning
Done: framing_programmable_making_automation_projects
Done: framing_numeracy_math_skills


---
## Step 7 — Structured Insight Extraction with Metadata Scope Signals

This step now runs in two model passes:

1. **Draft structured extraction**: convert synthesis into structured candidate
   insight objects with source topics.
2. **Metadata-aware finalization**: compute significant metadata lift for each
   draft candidate's supporting projects, then ask the model to revise only when
   the metadata helps scope the claim or avoid overgeneralization.

Metadata lift is optional guidance. It is not causal evidence and should not
force the model to mention a segment unless doing so improves accuracy.


In [14]:
# ── EXTERNAL-FACING INSIGHTS — structured JSON call with metadata lift ─────
# Pass 1: draft insights from synthesis.
# Pass 2: finalize those same insights with optional metadata scope signals.

OUTPUT_GROUP_KEY = "by_group"
REQUIRED_GROUP_VALUES = list(per_group_results.keys())

if not REQUIRED_GROUP_VALUES:
    raise RuntimeError("No synthesized group results are available; cannot build external-facing insights.")

synthesis_parts = []
if synthesis_cross:
    synthesis_parts.append(f"=== CROSS-GROUP ANALYSIS ===\n{synthesis_cross}")
for group_value, text in per_group_results.items():
    if text:
        synthesis_parts.append(f"=== {group_value} ===\n{text}")
synthesis_input = "\n\n".join(synthesis_parts)

INSIGHTS_SYSTEM = '''You are a senior program analyst at DonorsChoose.
You turn structured internal synthesis into clear, evidence-grounded insight objects for review by foundation leaders, corporate partners, policymakers, executives, and major donors.
You return ONLY valid JSON. No preamble, no explanation, no markdown fences.

Your job is not to write dramatic reveals.
Your job is to decide which claims are worth surfacing, state only what the evidence supports, explain the classroom or student-facing mechanism, and make clear how far the claim should be generalized.

Core quality bar:
- A generic insight is worse than no insight.
- An empty group is better than a group insight that merely restates the group label.
- You are rewarded for restraint, specificity, and clean source-topic grounding.
- Treat configured minimum counts as ceilings on ambition, not as targets to satisfy. The configured ranges describe what is permitted, not what is expected.
- Do not fill space. Do not satisfy a count target with weak material.

Evidence discipline:
- Use the structured synthesis and allowed source topic references as the evidence source.
- Treat synthesized Direct evidence, Interpretation, Scope, and Supporting topics as evidence units.
- Every insight must be grounded in the strongest matching source topics.
- Do not infer broad support if the synthesis only gives narrow support.
- Aim for 2 or more well-matched source_topics per insight when on-mechanism evidence supports it; cite a single topic only when no other topic in the evidence is genuinely on the same mechanism, classroom function, student condition, or operating pressure. Never pad with weakly related or adjacent topics to reach a count.
- Do not cite adjacent, illustrative, or merely thematically related topics.
- Do not merge adjacent synthesized findings unless they clearly support the same final insight.
- Do not use project-topic support counts as proof that every project expresses the full final claim.
- Do not use external current-events knowledge as evidence.

Downstream validation awareness:
- Each insight will later be checked against verified source topics.
- Insights with weak, loose, or nonexistent source topic grounding may be dropped.
- Write claims that would still stand if weakly related source topics were removed.
- A source topic should support the specific finding, not just the broad theme.
- If the only way to support a claim is to attach loose or adjacent source topics, omit the insight or narrow the claim.

Internal claim-role discipline:
- Before writing an insight, identify its claim role internally. Do not output this role as a field.
- Valid claim roles are: central mechanism, internal split, boundary correction, secondary strand, audience correction, and implementation implication.
- If the best role for a finding is merely surface confirmation of the group label, do not emit the insight.
- A central mechanism names the main classroom function or student-facing mechanism.
- An internal split separates two mechanisms that would otherwise be conflated.
- A boundary correction prevents an overclaim or misread.
- A secondary strand preserves a narrower but meaningful pattern.
- An audience correction changes how an external reader should interpret the area.
- An implementation implication names a practical requirement for funding, reporting, partnership design, policy understanding, or classroom use.

Insight selection:
- Prefer the smallest non-overlapping set of insights that captures the strongest supported patterns.
- Never add an insight just because there is room before the maximum count.
- Never add an insight merely because a minimum count exists.
- Do not split one mechanism into multiple insights to satisfy a minimum count.
- If two possible insights have the same practical implication, merge them or keep the stronger one.
- Do not force every insight into a looks-like-X-but-is-Y structure.
- Do not manufacture a reader mistake or surface-level misread.
- Avoid observations that merely restate what the group label already implies.
- For group-specific insights, ask whether the claim would still be informative if the group label were hidden. If not, omit it.
- A group-specific insight must add a mechanism, split, boundary, student condition, classroom routine, implementation implication, or non-obvious distinction beyond the group name.

Boundary finding promotion:
- Boundary findings should become standalone insights only when they materially change how an external audience should interpret the area, prevent a likely overclaim, identify a distinct funder-relevant need, or protect against a sensitive-context or equity-related misinterpretation.
- Otherwise, incorporate the boundary into scope_or_caveat of a related insight.

Substrate rule:
- Food, hygiene, clothing, seating, storage, sensory tools, calm spaces, headphones, basic supplies, and generic devices are common classroom substrate.
- Do not promote common substrate as a main insight unless the evidence shows a mechanism specific to the current group or cross-group pattern.
- If substrate appears only as generic classroom operating support, mention it only as evidence, background, or scope.
- A substrate insight must explain the specific classroom function or student-facing condition that makes the substrate analytically meaningful in this run.

Audience-facing voice:
- The reader does not know how the analysis was produced and should not need to. Final prose must read as an analyst describing classrooms and student conditions, not as a model describing what it found.
- Do not mention topics, source topics, clusters, synthesis, labels, models, prompts, or analytical machinery in title, finding, evidence_basis, scope_or_caveat, or why_it_matters.
- Do not write phrases such as "some topics describe," "the strongest topics," "topics focus on," "the synthesis links," "the evidence pairs," "within this group," "across topics," "multiple topics converge," "the group combines," or any sentence whose grammatical subject is a topic, cluster, or evidence unit.
- Do not paraphrase the forbidden phrases. "Multiple lines of evidence converge on X" is the same failure as "multiple topics converge on X." If the natural way to write the sentence requires naming the evidence apparatus, the sentence is wrong; rewrite it to describe the classroom reality directly.
- The grammatical subject of insight prose should be classrooms, students, teachers, requests, materials, routines, or conditions. Never the evidence apparatus.
- Default to students, classrooms, classroom routines, or classroom conditions as the subject when that is natural and evidence-faithful.
- Use teachers as the subject when the evidence is genuinely about what teachers request, build, adapt, manage, or describe.
- Do not write awkward agentless prose just to avoid teacher-first phrasing.
- The implication should connect back to students or classroom conditions when possible, but the sentence subject should be whatever makes the claim clearest.
- Write directly, warmly, and concretely. Warmth should come from classroom materials, routines, and student conditions, not from inspirational language.
- Avoid generic stakeholder language.
- Do not use em dashes.

Voice and claim strength:
- Vary sentence openings. Do not repeatedly start findings with the same stock phrase.
- Phrase the claim according to the evidence, not according to a fixed ladder.
- Use "classroom routines depend on..." only when recurring classroom operations or conditions are truly the claim.
- Use "students are positioned to..." only when the evidence directly connects materials or routines to a student-facing classroom function.
- Use "requests center on..." or "requests include..." sparingly, and only when a more concrete subject would be less clear.
- Use "teachers frame..." when the evidence is about teacher rationale, request language, adaptation, or classroom management choices.
- Use "is concentrated in..." only when a supplied scope signal or synthesis scope supports concentration.
- Avoid "proves," "shows," "demonstrates," or causal language for inferred outcomes, prevalence, or student experiences.

Current K-12 context discipline:
- You may recognize mechanisms relevant to classroom operations, budget pressure, technology use, behavior management, learning recovery, or basic needs only when the synthesis evidence directly supports them.
- Do not introduce named current-events frames such as ESSER cliff, post-pandemic recovery, AI, or cellphone policy unless those exact mechanisms are present in the supplied synthesis.
- Do not use trend or recency language without explicit time-comparison evidence.

Output style:
- Titles should be concrete and short. The title carries the substance of the claim, not the report structure.
- Findings should make one claim.
- evidence_basis should name concrete materials, routines, contexts, request patterns, or student conditions, and should add support logic rather than paraphrasing the finding.
- scope_or_caveat should prevent overgeneralization with one useful boundary.
- why_it_matters should name a concrete consequence: a decision that would change, an interpretation that would shift, a program design choice that would be informed, a measurement that would be reported differently, or an implementation requirement that would need to be met.'''.strip()

example_source_topics = []
example_rows = (
    labels_df[[GROUPBY_FIELD, "topic_id"]]
    .drop_duplicates()
    .head(3)
    .to_dict("records")
)
for row in example_rows:
    example_source_topics.append(f"{row[GROUPBY_FIELD]}|{int(row['topic_id'])}")
if not example_source_topics:
    for i, gv in enumerate(REQUIRED_GROUP_VALUES[:2], start=1):
        example_source_topics.append(f"{gv}|{i}")
example_source_topics_json = json.dumps(example_source_topics, ensure_ascii=False)

INSIGHT_SCHEMA_INSTRUCTIONS = f"""
INSIGHT STRUCTURE:
For every insight, use exactly these fields:

- title:
  A short statement, typically 5-10 words, naming the mechanism, split, boundary,
  or implication the insight is about. Center students, classrooms, classroom
  conditions, teacher actions, materials, routines, or school-day operations.

  Forbidden title openings and shapes:
    - "This group...", "This run...", "This category..."
    - "A separate strand...", "A second pattern...", "A meaningful share..."
    - "The main demand...", "The central pattern...", "A distinct slice..."
    - "Requests center on...", "X requests are...", "X requests support..."
    - Any title whose only content is the category name plus a generic verb,
      e.g. "Books support reading", "Lab requests center on inquiry",
      "Devices support learning access".

  A title must name something a reader could not already infer from the group
  label. If the title sentence would still be true with the group name replaced
  by a different category, the title is too generic; rewrite it to name the
  specific mechanism.

  Do not write a slogan. Do not force a contrast frame. Do not use "really
  about," "not just," "more than," "hidden," or "easy to miss." Do not merely
  paraphrase the group value.

- finding:
  1-3 sentences explaining one claim in direct, concrete language. State what
  the evidence supports. Match length to the complexity of the signal. Do not
  pad simple findings.
  A finding should name a classroom mechanism, internal split, boundary,
  student condition, operating pressure, implementation implication, or
  non-obvious distinction.
  Do not emit a finding that merely confirms what the group value already
  implies.

- evidence_basis:
  1-2 sentences naming the specific materials, classroom routines, student
  conditions, request patterns, or operating contexts that appear across the
  supporting evidence. Be concrete. Name actual items, activities, settings, or
  populations rather than restating the finding in different words.

  This field's distinct job is to anchor the claim in observable classroom
  facts and explain why the evidence belongs together. It is not a second
  statement of the finding, and it is not a description of the analysis. If
  the evidence_basis can be deleted without losing information already in the
  finding, the evidence_basis is doing the wrong job; rewrite it to add
  concrete classroom detail the finding does not contain.

  Do not list topics. Do not name the analytical process. Do not repeat
  category names. State what classrooms actually look like or contain in the
  cases supporting this claim.

- scope_or_caveat:
  Exactly 1 sentence naming the single most important constraint on
  interpretation for this claim. Do not list multiple caveats. Do not stack a
  centrality claim, a grade-band concentration, a category concentration, and
  a non-generalization warning into one field; pick the one that most changes
  how a reader should interpret the finding.

  Acceptable shapes: where the claim is strongest; what the reader should not
  overgeneralize to; what makes the claim narrower than its wording suggests;
  or what specific population, setting, or condition the pattern is anchored
  in.

  This is the preferred place to use optional metadata scope signals when they
  materially improve accuracy. If no caveat is materially needed, use the
  sentence to anchor the finding to the specific mechanism or evidence base
  where it is clearest. Do not write filler such as "This pattern may not
  apply everywhere" or "Further evidence may be needed." If you have nothing
  to say in this field, the insight is probably not insight-grade; reconsider
  whether to emit it.

- why_it_matters:
  Exactly 1 sentence naming a concrete consequence that follows from the
  finding being true. The consequence must be specific: a decision that would
  change, an interpretation that would shift, a program design choice that
  would be informed, a measurement that would be reported differently, or an
  implementation requirement that would need to be met. The test is whether a
  reader could act, classify, or decide differently because of the finding.

  Forbidden shapes: "This helps funders understand...", "This changes how
  external audiences interpret...", "This supports more precise framing...",
  "This points to the importance of...". These are not consequences; they are
  claims that a consequence exists. State the consequence itself.

  Avoid generic stakeholder language. Avoid call-to-action phrasing. Avoid
  donor-marketing register.

- source_topics:
  A list of the strongest grounding topics for the insight. Use only topics
  that directly support the specific claim, not topics that merely share the
  broad category.

  Usually 2-4 topics is appropriate, but do not force a fixed count. A single
  supporting topic is a warning sign: the insight is acceptable only when the
  claim is explicitly narrow ("a distinct strand of...", "in this slice...")
  and a second on-mechanism topic genuinely does not exist in the evidence. A
  broad claim resting on one topic, or a claim that requires adjacent or
  thematically related topics to reach two, is not insight-grade; narrow the
  claim or omit the insight.

  Never pad with weak or adjacent matches to reach a count.

  Required format:
    - each item must be a string
    - each string must be exactly "<group_value>|<topic_id>"
    - group_value must exactly match one of the group values shown in the synthesis
    - topic_id must be the integer topic number for that group
  Example for this run: {example_source_topics_json}

SOURCE_TOPIC RULES:
- Prefer source_topics that are explicitly named in Supporting topics lines.
- If a synthesized finding names multiple supporting topics but only some
  directly support your final claim, include only the strongest matches.
- If a topic is illustrative but not necessary to support the claim, leave it out.
- Do not use labels, prose, section names, or invented group names in place of group_value.
- source_topics must be a list of strings only, not objects.
- Do not include any extra fields beyond: title, finding, evidence_basis,
  scope_or_caveat, why_it_matters, source_topics.

STYLE:
- Direct, warm, concrete, and human.
- Not academic.
- Not corporate boilerplate.
- Not donor-marketing fluff.
- No named places.
- No em dashes.
- No mention of topics, source topics, models, clusters, prompts, synthesis,
  or analytical machinery in title, finding, evidence_basis, scope_or_caveat,
  or why_it_matters.
- Do not write phrases such as "some topics describe," "the strongest topics,"
  "topics focus on," "the synthesis links," "the evidence pairs," or "within
  this group."
- Do not manufacture a "why this is easy to miss" explanation.
- Do not use "may suggest" or "could indicate," but do use scope language when support is narrow.
- No filler.
""".strip()

DRAFT_PROMPT = '''Convert the synthesis below into structured external-facing insight objects.

Run context:
{RUN_CONTEXT_BLOCK}

Strategic/context caveats:
{STRATEGIC_CONTEXT_BLOCK}

Allowed source topic references:
The canonical source_topics format is a pipe-string: <group_value>|<topic_id>.
Use only IDs from the lookup below. Do not invent IDs. Do not emit dictionaries.
{ALLOWED_TOPIC_REFERENCES}

Structured synthesis:
{SYNTHESIS_INPUT}

Insight writing rules:
{INSIGHT_SCHEMA_INSTRUCTIONS}

Required JSON shape:
{{
  "key_insights": [
    {{
      "title": "...",
      "finding": "...",
      "evidence_basis": "...",
      "scope_or_caveat": "...",
      "why_it_matters": "...",
      "source_topics": ["<group_value>|<topic_id>", "<group_value>|<topic_id>"]
    }}
  ],
  "{OUTPUT_GROUP_KEY}": {{
    "<exact group value>": [
      {{
        "title": "...",
        "finding": "...",
        "evidence_basis": "...",
        "scope_or_caveat": "...",
        "why_it_matters": "...",
        "source_topics": ["<group_value>|<topic_id>"]
      }}
    ]
  }}
}}

Count guidance:
- key_insights: return 0 to {CROSS_MAX_INSIGHTS}. The minimum {CROSS_MIN_INSIGHTS} is permitted, not expected. Returning fewer, including zero, is the correct choice when the synthesis does not clearly support more strong, non-overlapping insights.
- each group under {OUTPUT_GROUP_KEY}: return 0 to {PER_GROUP_MAX_INSIGHTS}. The minimum {PER_GROUP_MIN_INSIGHTS} is permitted, not expected. An empty list is the correct output when a group has no specific mechanism, split, boundary, or implication beyond what the group label already implies.
- An empty group output is preferred over a generic group output. A generic group insight is a quality failure, not acceptable coverage.
- You are rewarded for restraint.
- Do not add weak insights just to satisfy a minimum.
- Do not split one mechanism into multiple insights to satisfy a minimum.
- Do not create a group insight whose main claim is already implied by the group value.
- Do not create a surface-confirmation insight merely to provide coverage.

Group coverage:
- Include every group key exactly as listed here: {REQUIRED_GROUP_VALUES_JSON}.
- Group coverage means including every group key, not producing at least one insight per group.
- Empty group lists are valid and expected when the synthesis does not support a specific, non-obvious, well-grounded claim.
- Do not treat an empty group list as a failure.
- A generic group summary is worse than no group insight.

Source topic rules:
- source_topics must be pipe-string values only.
- Use only source topics listed in Allowed source topic references.
- Copy group values exactly.
- Aim for 2 or more directly supporting topics per insight when the synthesis evidence supports it. A single supporting topic is acceptable only when no other topic in the synthesis is on the same mechanism. Never include loose, adjacent, or weakly related topics to reach a count.
- If a claim needs weak or adjacent source topics to seem supported, narrow the claim or omit the insight.
- If a synthesis finding cites a cluster indirectly through member topics, cite only the member topic pipe strings.
- Never cite cluster IDs.

Return valid JSON only.'''.format(
    RUN_CONTEXT_BLOCK=(
        f"- Grouping field: {GROUPBY_FIELD}\n"
        f"- Group description: {GROUP_DESCRIPTION}\n"
        f"- Required group count: {len(REQUIRED_GROUP_VALUES)}"
    ),
    STRATEGIC_CONTEXT_BLOCK=(
        f"- Strategic loop enabled: {STRATEGIC_LOOP_ENABLED}\n"
        f"- Strategic run metadata: {json.dumps({k: STRATEGIC_RUN_META.get(k) for k in ['strategic_area_id', 'strategic_area_label', 'split_id', 'groupby_fields', 'is_strategic_injected_tag']}, ensure_ascii=False)}"
    ),
    ALLOWED_TOPIC_REFERENCES="\n".join(
        f"- {row[GROUPBY_FIELD]}|{int(row['topic_id'])}: {str(row.get('proposed_label', '') or '').strip()}. {str(row.get('description', '') or '').strip()}"
        for _, row in labels_df[[GROUPBY_FIELD, "topic_id", "proposed_label", "description"]]
            .drop_duplicates([GROUPBY_FIELD, "topic_id"])
            .iterrows()
    ),
    SYNTHESIS_INPUT=synthesis_input,
    INSIGHT_SCHEMA_INSTRUCTIONS=INSIGHT_SCHEMA_INSTRUCTIONS,
    OUTPUT_GROUP_KEY=OUTPUT_GROUP_KEY,
    REQUIRED_GROUP_VALUES_JSON=json.dumps(REQUIRED_GROUP_VALUES, ensure_ascii=False),
    CROSS_MIN_INSIGHTS=CROSS_MIN_INSIGHTS,
    CROSS_MAX_INSIGHTS=CROSS_MAX_INSIGHTS,
    PER_GROUP_MIN_INSIGHTS=PER_GROUP_MIN_INSIGHTS,
    PER_GROUP_MAX_INSIGHTS=PER_GROUP_MAX_INSIGHTS,
).strip()

def _normalize_insights_response(raw_obj, output_group_key, required_group_values):
    if not isinstance(raw_obj, dict):
        raise ValueError("Top-level insights response is not a JSON object.")

    raw_key = raw_obj.get("key_insights", [])
    raw_by_group = raw_obj.get(output_group_key, {})

    if not isinstance(raw_key, list):
        raise ValueError("key_insights is not a list.")
    if not isinstance(raw_by_group, dict):
        raise ValueError(f"{output_group_key} is not an object.")

    normalized = {
        "key_insights": [
            normalize_insight(i, required_group_values=required_group_values)
            for i in raw_key
        ],
        output_group_key: {},
    }

    missing_group_values = [g for g in required_group_values if g not in raw_by_group]
    if missing_group_values:
        raise ValueError(f"Model omitted required group values: {missing_group_values}")

    extra_group_values = [g for g in raw_by_group.keys() if g not in required_group_values]
    if extra_group_values:
        print(f"WARNING: extra group values returned and ignored: {extra_group_values}")

    for group_value in required_group_values:
        items = raw_by_group.get(group_value, [])
        if not isinstance(items, list):
            raise ValueError(f"{output_group_key}['{group_value}'] is not a list.")
        normalized[output_group_key][group_value] = [
            normalize_insight(i, required_group_values=required_group_values)
            for i in items
        ]
    return normalized

def _validate_insight_counts(data, output_group_key):
    # Upper-bound only. The prompts intentionally permit fewer than minimum
    # counts, including empty per-group lists, when evidence does not support
    # more. Below-minimum is logged as a warning but does not fail validation.
    #
    # Required shape and required group keys are still enforced elsewhere in
    # _normalize_insights_response.

    key_items = data.get("key_insights", [])
    group_items = data.get(output_group_key, {})

    n_key = len(key_items)
    if n_key > CROSS_MAX_INSIGHTS:
        raise ValueError(
            f"Expected at most {CROSS_MAX_INSIGHTS} key insights, got {n_key}"
        )

    over_max_groups = {
        group_value: len(items)
        for group_value, items in group_items.items()
        if len(items) > PER_GROUP_MAX_INSIGHTS
    }
    if over_max_groups:
        raise ValueError(
            f"Expected at most {PER_GROUP_MAX_INSIGHTS} insights per group value, "
            f"got: {over_max_groups}"
        )

    # Soft signal for analysts: below-minimum is permitted but worth noting so
    # intentional abstention can be distinguished from weak model output.
    if n_key < CROSS_MIN_INSIGHTS:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_COUNT_BELOW_MIN",
            f"key_insights returned {n_key}, below configured min {CROSS_MIN_INSIGHTS}",
            context={
                "section": "key_insights",
                "count": n_key,
                "min": CROSS_MIN_INSIGHTS,
            },
        )

    under_min_groups = {
        group_value: len(items)
        for group_value, items in group_items.items()
        if len(items) < PER_GROUP_MIN_INSIGHTS
    }
    if under_min_groups:
        # Empty or under-min groups are now an intended output under the
        # abstention-first prompts. Log to console for visibility, but do not
        # write to warnings.jsonl since these are not quality failures.
        print(
            f"INFO: {len(under_min_groups)} group(s) returned below configured min "
            f"{PER_GROUP_MIN_INSIGHTS}: {sorted(under_min_groups.keys())}"
        )

def _iter_all_insights(data, output_group_key):
    for idx, item in enumerate(data.get("key_insights", []), start=1):
        yield f"KI_{idx:03d}", "key_insights", None, item
    for group_value, items in data.get(output_group_key, {}).items():
        for idx, item in enumerate(items, start=1):
            yield f"BG_{group_value}_{idx:03d}", output_group_key, group_value, item


def _fail_current_run(stage: str, code: str, message: str, *, error: Exception | None = None, context: dict | None = None) -> None:
    """Record a friendly run-level failure artifact, then fail this combo.

    In the strategic sweep, the subprocess exits non-zero and the runner records
    this combo as failed, then continues to the next combo.
    """
    payload = {
        "status": "failed",
        "stage": stage,
        "code": code,
        "message": message,
        "error": f"{type(error).__name__}: {error}" if error is not None else None,
        "warnings_path": str(WARNINGS_PATH),
        "llm": {
            "max_retries": MAX_RETRIES,
            "service_tier": LLM_SERVICE_TIER,
            "flex_attempts": LLM_FLEX_ATTEMPTS,
            "fallback_service_tier": LLM_FALLBACK_SERVICE_TIER,
            "timeout_seconds": LLM_TIMEOUT_SECONDS,
            "retry_delay_seconds": LLM_RETRY_DELAY,
            "flex_retry_delay_seconds": LLM_FLEX_RETRY_DELAY,
        },
        "context": context or {},
    }
    append_warning(
        WARNINGS_PATH,
        "03_insights_generation",
        code,
        message,
        severity="error",
        context={k: v for k, v in payload.items() if k not in {"status", "message"}},
    )
    try:
        write_json(OUT("insights", f"run_failed_{stage}.json"), payload)
    except Exception as artifact_error:
        print(f"WARNING: could not write run failure artifact for {stage}: {artifact_error}")

    print("\nRUN FAILED")
    print(f"Stage: {stage}")
    print(f"Code: {code}")
    print(message)
    if error is not None:
        print(f"Underlying error: {type(error).__name__}: {error}")
    print(f"Warnings: {WARNINGS_PATH}")
    raise RuntimeError(
        f"{stage} failed after all retries. This combo is marked failed; "
        f"the strategic sweep runner will continue to the next combo. "
        f"See {WARNINGS_PATH}."
    )


# Pass 1: draft structured insights.
draft_resp = None
last_draft_error = None
for _attempt in range(MAX_RETRIES + 1):
    try:
        draft_resp = chat_completion_with_tier_fallback(
            client=client,
            model=MODEL_SYNTHESIS,
            messages=[
                {"role": "system", "content": INSIGHTS_SYSTEM},
                {"role": "user", "content": DRAFT_PROMPT},
            ],
            response_format={"type": "json_object"},
            service_tier=LLM_SERVICE_TIER,
            flex_attempts=LLM_FLEX_ATTEMPTS,
            fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
            timeout_seconds=LLM_TIMEOUT_SECONDS,
            flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
        )
        break
    except Exception as e:
        last_draft_error = e
        if _attempt < MAX_RETRIES:
            if LLM_RETRY_DELAY > 0:
                time.sleep(LLM_RETRY_DELAY)
            continue

if draft_resp is None:
    _fail_current_run(
        "insights_draft",
        "INSIGHTS_DRAFT_API_FAILED",
        "Draft insight generation failed after all retries; this run cannot produce final insights.",
        error=last_draft_error,
    )

try:
    draft_raw_json = strip_json_fences(draft_resp.choices[0].message.content)
    draft_insights_raw = json.loads(draft_raw_json)
    draft_insights_data = _normalize_insights_response(
        draft_insights_raw,
        OUTPUT_GROUP_KEY,
        REQUIRED_GROUP_VALUES,
    )
    _validate_insight_counts(draft_insights_data, OUTPUT_GROUP_KEY)
except Exception as e:
    _fail_current_run(
        "insights_draft",
        "INSIGHTS_DRAFT_PARSE_OR_VALIDATION_FAILED",
        "Draft insight generation returned a response that could not be parsed or validated; this run cannot produce final insights.",
        error=e,
        context={"raw_response_preview": (locals().get("draft_raw_json") or "")[:2000]},
    )

write_json(
    OUT("insights", "insights_candidates_draft_pre_metadata.json"),
    {
        "key_insights": [
            project_insight_for_saved_candidates(i)
            for i in draft_insights_data["key_insights"]
        ],
        OUTPUT_GROUP_KEY: {
            group_value: [
                project_insight_for_saved_candidates(i)
                for i in items
            ]
            for group_value, items in draft_insights_data[OUTPUT_GROUP_KEY].items()
        },
    },
)

# Compute candidate-level metadata scope signals from draft source topics.
BRIDGE_LOOKUP_FOR_LIFT = build_bridge_lookup(project_topic_bridge_df)

METADATA_LIFT_CONTEXT = build_metadata_lift_context(
    run_df=df,
    dimensions=METADATA_LIFT_DIMENSIONS,
    project_id_col="project_id",
) if METADATA_LIFT_ENABLED else None

candidate_packages = []
metadata_lift_rows = []

for candidate_id, section, group_value, item in _iter_all_insights(draft_insights_data, OUTPUT_GROUP_KEY):
    support_ids = candidate_support_project_ids_from_source_topics(
        item.get("source_topics", []),
        groupby_field=GROUPBY_FIELD,
        bridge_lookup=BRIDGE_LOOKUP_FOR_LIFT,
    )

    if METADATA_LIFT_ENABLED and METADATA_LIFT_CONTEXT is not None:
        lift_df = compute_metadata_lift(
            support_project_ids=support_ids,
            context=METADATA_LIFT_CONTEXT,
            thresholds=METADATA_LIFT_THRESHOLDS,
        )
    else:
        lift_df = pd.DataFrame()

    if not lift_df.empty:
        lift_df = lift_df.copy()
        lift_df.insert(0, "candidate_id", candidate_id)
        lift_df.insert(1, "section", section)
        lift_df.insert(2, "group_value", group_value)
        metadata_lift_rows.append(lift_df)

    metadata_lift_block = format_metadata_lift_facts(lift_df)
    candidate_packages.append({
        "candidate_id": candidate_id,
        "section": section,
        "group_value": group_value,
        "draft_insight": project_insight_for_saved_candidates(item),
        "supporting_project_count_for_lift": len(support_ids),
        "metadata_scope_signals": metadata_lift_block,
    })

metadata_lift_audit_df = (
    pd.concat(metadata_lift_rows, ignore_index=True)
    if metadata_lift_rows else
    pd.DataFrame(columns=[
        "candidate_id", "section", "group_value", "dimension", "value",
        "insight_project_count", "run_project_count", "insight_total_projects",
        "run_total_projects", "insight_share", "run_share", "difference_pp",
        "abs_difference_pp", "lift", "cohens_h",
    ])
)
metadata_lift_audit_df.to_csv(
    OUT("insights", "metadata_lift_facts_for_step7.csv"),
    index=False,
)

print(
    f"Metadata scope signals prepared for "
    f"{sum(bool(p.get('metadata_scope_signals')) for p in candidate_packages):,} "
    f"of {len(candidate_packages):,} draft candidates."
)

# Pass 2: finalize insight language with optional metadata scope signals.
HAS_METADATA_SCOPE_SIGNALS = any(
    bool(p.get("metadata_scope_signals"))
    for p in candidate_packages
)

if HAS_METADATA_SCOPE_SIGNALS:
    FINALIZE_SYSTEM = '''You are finalizing draft DonorsChoose insight objects using optional metadata scope signals.
This is a wording and scope pass only.
You return ONLY valid JSON. No preamble, no explanation, no markdown fences.

Non-negotiable preservation rules:
- Do not generate new insights.
- Do not remove insights.
- Do not split or merge insights.
- Do not change the number, order, sections, or group buckets.
- Do not add, remove, reorder, or edit source_topics.
- Do not change the field structure or add fields.
- Preserve each insight's core claim unless the draft clearly overstates scope.
- Preserve each field's role: do not move content from finding into evidence_basis, do not expand scope_or_caveat into a multi-caveat list, do not soften why_it_matters into generic stakeholder language.
- Preserve the length envelope. Do not lengthen any field. If a field can be shortened without losing precision, shorten it; do not pad to match the draft.

Metadata scope discipline:
- Metadata is not causal evidence.
- Metadata must not create a new insight.
- Use metadata only when it helps prevent overbroad wording.
- Metadata should usually affect scope_or_caveat.
- Only revise finding, evidence_basis, or why_it_matters if the draft clearly overstates scope.
- If metadata does not materially improve an insight, preserve the draft wording.
- Treat a signal as materially narrowing when supporting projects are at least roughly twice as concentrated in one segment as in the run as a whole.
- For binary or yes/no dimensions, treat a signal as materially narrowing when the supporting projects' share differs from the run baseline by at least 15 percentage points, even if the multiplier is below 2x.
- Do not mention every metadata fact.
- Do not reflect metadata that simply restates the topic of the insight.
- If the metadata does not materially change the scope, leave the insight scoped by the source topics only.
- scope_or_caveat must remain exactly one sentence and must name only the most important boundary.
- Do not turn scope_or_caveat into a list of segment facts.

Field discipline:
- title should remain short, concrete, and free of report-structure throat-clearing.
- finding should remain one claim.
- evidence_basis should name concrete materials, routines, contexts, request patterns, student conditions, or classroom uses. It should not mention topics, source topics, clusters, synthesis, or analytical machinery.
- evidence_basis should not paraphrase the finding.
- scope_or_caveat should prevent overgeneralization with one useful boundary.
- why_it_matters should state a concrete consequence: a decision that would change, an interpretation that would shift, a program design choice that would be informed, a measurement that would be reported differently, or an implementation requirement that would need to be met.
- Do not make fields longer unless needed to correct overbroad scope.

Strategic-run discipline:
- If the run is a strategic-loop run, use the strategic area only to avoid overbroad wording.
- If groups are strategic injected tags, describe them as tag-defined slices within the strategic area, not mutually exclusive market segments.
- Do not introduce the strategic area into title or finding unless it is necessary for accuracy.

Audience-facing voice:
- Write as if the reader does not know the synthesis, topic, cluster, model, prompt, or source-topic process.
- Do not mention topics, clusters, synthesis, source topics, labels, models, prompts, or analytical machinery in title, finding, evidence_basis, scope_or_caveat, or why_it_matters.
- Do not write phrases such as "some topics describe," "the strongest topics," "topics focus on," "the synthesis links," "the evidence pairs," or "within this group."
- Do not paraphrase the forbidden phrases. "Multiple lines of evidence converge on X" is the same failure as "multiple topics converge on X." Rewrite to describe the classroom reality directly.
- Translate all evidence into direct audience-facing prose about requests, classrooms, students, materials, routines, conditions, or implementation needs.
- Default to students, classrooms, classroom routines, or classroom conditions as the subject when natural and evidence-faithful.
- Use teachers as the subject when the evidence is about teacher requests, framing, adaptation, or management.
- Preserve the draft's claim-strength level unless scope narrowing requires more cautious wording.
- Vary sentence openings. Do not repeatedly start findings with the same stock phrase.
- Use "is concentrated in..." only when supplied metadata or draft scope supports concentration.
- Do not use causal, prevalence, trend, or recency language unless the draft already had explicit evidence for it.
- Do not use em dashes.

Example of appropriate metadata narrowing:
- Before: "Requests connect flexible seating to focus and regulation across classrooms."
- Metadata signal: "Grade band: Elementary is 63% of supporting projects vs 28% of this run (+35pp, 2.3x)."
- After: "Requests connect flexible seating to focus and regulation, with the strongest concentration in elementary classrooms."'''.strip()

    FINALIZE_PROMPT = '''Finalize the draft insight objects using optional metadata scope signals.

Run context:
{RUN_CONTEXT_BLOCK}

Strategic/context caveats:
{STRATEGIC_CONTEXT_BLOCK}

Draft candidates with optional metadata scope signals:
{CANDIDATE_PACKAGES_JSON}

Return the same top-level structure as the drafts:
{{
  "key_insights": [/* same number and order as draft key_insights */],
  "{OUTPUT_GROUP_KEY}": {{
    "<group value>": [/* same number and order as draft group insights */]
  }}
}}

Each returned insight must keep exactly these fields:
- title
- finding
- evidence_basis
- scope_or_caveat
- why_it_matters
- source_topics

Field rules:
- Preserve the draft unless metadata materially narrows or corrects the scope.
- Keep scope_or_caveat to exactly one sentence.
- Do not add process language, analytical machinery, or topic-model terminology.
- Do not lengthen evidence_basis or scope_or_caveat unless needed to correct overbroad wording.
- source_topics must be copied exactly from the draft for each insight.

Return valid JSON only.'''.format(
        RUN_CONTEXT_BLOCK=(
            f"- Grouping field: {GROUPBY_FIELD}\n"
            f"- Group description: {GROUP_DESCRIPTION}"
        ),
        STRATEGIC_CONTEXT_BLOCK=(
            f"- Strategic loop enabled: {STRATEGIC_LOOP_ENABLED}\n"
            f"- Strategic run metadata: {json.dumps({k: STRATEGIC_RUN_META.get(k) for k in ['strategic_area_id', 'strategic_area_label', 'split_id', 'groupby_fields', 'is_strategic_injected_tag']}, ensure_ascii=False)}"
        ),
        CANDIDATE_PACKAGES_JSON=json.dumps(candidate_packages, ensure_ascii=False, indent=2),
        OUTPUT_GROUP_KEY=OUTPUT_GROUP_KEY,
    ).strip()

    final_resp = None
    last_finalize_error = None
    for _attempt in range(MAX_RETRIES + 1):
        try:
            final_resp = chat_completion_with_tier_fallback(
                client=client,
                model=MODEL_SYNTHESIS,
                messages=[
                    {"role": "system", "content": FINALIZE_SYSTEM},
                    {"role": "user", "content": FINALIZE_PROMPT},
                ],
                response_format={"type": "json_object"},
                service_tier=LLM_SERVICE_TIER,
                flex_attempts=LLM_FLEX_ATTEMPTS,
                fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
                timeout_seconds=LLM_TIMEOUT_SECONDS,
                flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
            )
            break
        except Exception as e:
            last_finalize_error = e
            if _attempt < MAX_RETRIES:
                if LLM_RETRY_DELAY > 0:
                    time.sleep(LLM_RETRY_DELAY)
                continue

    if final_resp is None:
        _fail_current_run(
            "insights_finalize",
            "INSIGHTS_FINALIZE_API_FAILED",
            "Insight finalization failed after all retries; this run cannot produce final insights.",
            error=last_finalize_error,
        )

    try:
        final_raw_json = strip_json_fences(final_resp.choices[0].message.content)
        insights_data = _normalize_insights_response(
            json.loads(final_raw_json),
            OUTPUT_GROUP_KEY,
            REQUIRED_GROUP_VALUES,
        )
        _validate_insight_counts(insights_data, OUTPUT_GROUP_KEY)
    except Exception as e:
        _fail_current_run(
            "insights_finalize",
            "INSIGHTS_FINALIZE_PARSE_OR_VALIDATION_FAILED",
            "Insight finalization returned a response that could not be parsed or validated; this run cannot produce final insights.",
            error=e,
            context={"raw_response_preview": (locals().get("final_raw_json") or "")[:2000]},
        )
else:
    print("No metadata lift facts passed Step 7 thresholds; skipping metadata-scope refinement pass.")
    insights_data = draft_insights_data

HARD_STYLE_PATTERNS = [
    r"—",
    r"\bnot just\b",
    r"\bat its core\b",
    r"\bthis speaks to\b",
    r"\bit is clear that\b",
    r"\bperhaps most importantly\b",
    r"\bin a world where\b",
    r"\bsome topics describe\b",
    r"\bthe strongest topics\b",
    r"\btopics focus on\b",
    r"\btopics describe\b",
    r"\bthe synthesis links\b",
    r"\bthe evidence pairs\b",
    r"\bwithin this group\b",
    r"\ba separate strand\b",
    r"\ba second pattern\b",
    r"\ba meaningful share\b",
]

SOFT_STYLE_PATTERNS = [
    r"\bleverage\b",
    r"\butilize\b",
    r"\bfacilitate\b",
    r"\bimpactful\b",
    r"\bstakeholders\b",
    r"\bend users\b",
    r"\bbeneficiaries\b",
    r"\ba significant number of\b",
]

def _pattern_hits(text, patterns):
    return [p for p in patterns if re.search(p, text, flags=re.IGNORECASE)]

for _, section, group_value, item in _iter_all_insights(insights_data, OUTPUT_GROUP_KEY):
    combined = " ".join(
        str(item.get(k, "") or "")
        for k in ["title", "finding", "evidence_basis", "scope_or_caveat", "why_it_matters"]
    )

    hard_hits = _pattern_hits(combined, HARD_STYLE_PATTERNS)
    soft_hits = _pattern_hits(combined, SOFT_STYLE_PATTERNS)

    if hard_hits:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_HARD_STYLE_VIOLATION",
            "Insight contains hard-banned DonorsChoose style pattern",
            context={
                "section": section,
                "group_value": group_value,
                "title": item.get("title", ""),
                "violations": hard_hits,
            },
        )
        print(f"HARD STYLE WARNING: {item.get('title', '')[:80]} | {hard_hits}")

    if soft_hits:
        append_warning(
            WARNINGS_PATH,
            "03_insights_generation",
            "INSIGHT_SOFT_STYLE_WARNING",
            "Insight contains soft-flagged DonorsChoose style pattern",
            context={
                "section": section,
                "group_value": group_value,
                "title": item.get("title", ""),
                "violations": soft_hits,
            },
        )
        print(f"SOFT STYLE WARNING: {item.get('title', '')[:80]} | {soft_hits}")

insights_candidates_for_save = {
    "key_insights": [
        project_insight_for_saved_candidates(i)
        for i in insights_data["key_insights"]
    ],
    OUTPUT_GROUP_KEY: {
        group_value: [
            project_insight_for_saved_candidates(i)
            for i in items
        ]
        for group_value, items in insights_data[OUTPUT_GROUP_KEY].items()
    },
}

# Mid-cell write required by spec: post-normalize_insight(), pre-verification.
write_json(OUT("insights", "insights_candidates.json"), insights_candidates_for_save)

print(
    f"Structured insight extraction complete: "
    f"{len(insights_data['key_insights'])} key insights, "
    f"{sum(len(v) for v in insights_data[OUTPUT_GROUP_KEY].values())} by-group insights, "
    f"{len(metadata_lift_audit_df)} metadata lift facts passed Step 7 thresholds."
)

Metadata scope signals prepared for 24 of 24 draft candidates.
HARD STYLE WARNING: Gardens function as recurring life science labs | ['\\bnot just\\b']
HARD STYLE WARNING: Microscopes anchor a specific life science routine | ['\\bnot just\\b']
HARD STYLE WARNING: Young students enter coding through physical robots | ['\\bnot just\\b']
HARD STYLE WARNING: Circuit lessons depend on parts students can assemble | ['\\bnot just\\b']
HARD STYLE WARNING: Prototype work needs the full design-to-production chain | ['\\bnot just\\b']
HARD STYLE WARNING: Robotics teams run on season-long operations | ['\\bnot just\\b']
Structured insight extraction complete: 6 key insights, 18 by-group insights, 115 metadata lift facts passed Step 7 thresholds.


---
## Step 8 — Topic Verification

Verifies that each claimed source topic genuinely supports its insight. Topics
that are too broad, indirect, or adjacent are dropped.

Verification strategy:
- **Cross-group insights** — always verified.
- **By-group insights** — verified only when they cite ≥ `by_group_min_source_topics`
  source topics; narrow local findings with 1–2 cited topics pass through without
  an additional API call.

In [15]:
# ── Topic verification ─────────────────────────────────────────────────────
# Verify source-topic grounding for final insights.
# Strategy:
# - always verify key_insights
# - verify by-group insights only when they cite 3+ source topics
#   (broad claims are more likely to overclaim support than narrow local ones)

VERIFY_SYSTEM = (
    "You are validating whether cited topics directly support an insight. "
    "Be strict. Keep a topic only if its label and description directly support "
    "the insight's title, finding, and evidence_basis. "
    "Ignore any practical implication or downstream strategy that is not part of the finding. "
    "If support is broad, indirect, adjacent, generic substrate, or only loosely related, drop it. "
    "Prefer false negatives to false positives. "
    "Return only valid JSON."
)

VERIFY_BY_GROUP_MIN_SOURCE_TOPICS = VERIFY_CFG["by_group_min_source_topics"]

# Always verify cross-group/key insights
insights_data["key_insights"], key_verify_stats = _verify_insight_list(
    insights_data.get("key_insights", []),
    labels_df=labels_df,
    groupby_field=GROUPBY_FIELD,
    required_group_values=REQUIRED_GROUP_VALUES,
    client=client,
    model_verify=MODEL_VERIFY,
    system_prompt=VERIFY_SYSTEM,
    warnings_path=WARNINGS_PATH,
    min_source_topics_to_verify=1,
    max_retries=MAX_RETRIES,
    retry_delay_seconds=LLM_RETRY_DELAY,
    service_tier=LLM_SERVICE_TIER,
    flex_attempts=LLM_FLEX_ATTEMPTS,
    fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
    timeout_seconds=LLM_TIMEOUT_SECONDS,
    flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
)

group_verify_stats = {}
for cat in insights_data.get(OUTPUT_GROUP_KEY, {}):
    verified_items, stats = _verify_insight_list(
        insights_data[OUTPUT_GROUP_KEY][cat],
        labels_df=labels_df,
        groupby_field=GROUPBY_FIELD,
        required_group_values=REQUIRED_GROUP_VALUES,
        client=client,
        model_verify=MODEL_VERIFY,
        system_prompt=VERIFY_SYSTEM,
        warnings_path=WARNINGS_PATH,
        min_source_topics_to_verify=VERIFY_BY_GROUP_MIN_SOURCE_TOPICS,
        max_retries=MAX_RETRIES,
        retry_delay_seconds=LLM_RETRY_DELAY,
        service_tier=LLM_SERVICE_TIER,
        flex_attempts=LLM_FLEX_ATTEMPTS,
        fallback_service_tier=LLM_FALLBACK_SERVICE_TIER,
        timeout_seconds=LLM_TIMEOUT_SECONDS,
        flex_retry_delay_seconds=LLM_FLEX_RETRY_DELAY,
    )
    insights_data[OUTPUT_GROUP_KEY][cat] = verified_items
    group_verify_stats[cat] = stats

total_group_changed = sum(v["changed_count"] for v in group_verify_stats.values())
total_group_zeroed = sum(v["dropped_to_zero_count"] for v in group_verify_stats.values())
total_group_topics_before = sum(v["topics_before"] for v in group_verify_stats.values())
total_group_topics_after = sum(v["topics_after"] for v in group_verify_stats.values())

print("Verification complete.")
print(
    f"Key insights: {key_verify_stats['insight_count']} insights | "
    f"{key_verify_stats['changed_count']} changed | "
    f"{key_verify_stats['dropped_to_zero_count']} dropped to zero topics | "
    f"{key_verify_stats['topics_before']} -> {key_verify_stats['topics_after']} source topics"
)
print(
    f"By-group insights: {sum(v['insight_count'] for v in group_verify_stats.values())} insights | "
    f"{total_group_changed} changed | "
    f"{total_group_zeroed} dropped to zero topics | "
    f"{total_group_topics_before} -> {total_group_topics_after} source topics | "
    f"verified only when source_topics >= {VERIFY_BY_GROUP_MIN_SOURCE_TOPICS}"
)

Adjusted source_topics: Technology requests split between access and engineering | 4 -> 0
Adjusted source_topics: Robotics teams run on season-long operations | 3 -> 2
Adjusted source_topics: Models and specimens make invisible science visible | 3 -> 0
Verification complete.
Key insights: 6 insights | 1 changed | 1 dropped to zero topics | 23 -> 19 source topics
By-group insights: 18 insights | 2 changed | 1 dropped to zero topics | 34 -> 30 source topics | verified only when source_topics >= 3


---
## Step 9 — Evidence Tables

Expands each verified insight into a flat summary row (`insights_flat.csv`) and
a per-topic support detail row (`insight_topic_support.csv`). Projects are ranked
by combined topic_share across all of an insight's verified topics so Looker
links surface the most representative essays.

In [ ]:
# ── VERIFIED INSIGHT SUPPORT TABLES ────────────────────────────────────────

assert "project_topic_bridge_df" in dir() and not project_topic_bridge_df.empty, (
    "project_topic_bridge_df missing. Ensure the bridge-build cell ran successfully."
)

BRIDGE_LOOKUP = build_bridge_lookup(project_topic_bridge_df)

label_index = build_label_index(
    labels_df,
    groupby_field=GROUPBY_FIELD,
    warnings_path=WARNINGS_PATH,
)

insights_flat_df, insight_topic_support_df = build_verified_insight_tables(
    insights_data,
    OUTPUT_GROUP_KEY,
    groupby_field=GROUPBY_FIELD,
    bridge_lookup=BRIDGE_LOOKUP,
    label_index=label_index,
    run_id=RUN_ID,
    top_project_id_limit=CSV_MAX_IDS_PER_INSIGHT,
)

# Attach strategic-run metadata for downstream multi-run review/HTML work.
for frame in [insights_flat_df, insight_topic_support_df]:
    frame["strategic_loop_enabled"] = STRATEGIC_LOOP_ENABLED
    frame["strategic_area_id"] = STRATEGIC_RUN_META.get("strategic_area_id")
    frame["strategic_area_label"] = STRATEGIC_RUN_META.get("strategic_area_label")
    frame["split_id"] = STRATEGIC_RUN_META.get("split_id")
    frame["resolved_groupby_field"] = GROUPBY_FIELD

insights_flat_df.to_csv(OUT("insights", "insights_flat.csv"), index=False)

# Candidate-specific name for sweep review.
insight_topic_support_df.to_csv(
    OUT("insights", "insight_topic_support_candidates.csv"),
    index=False,
)

# Canonical name for compatibility with any downstream manifest/report logic.
insight_topic_support_df.to_csv(
    OUT("insights", "insight_topic_support.csv"),
    index=False,
)

print(
    f"Verified insight support tables complete: "
    f"{len(insights_flat_df):,} insights and "
    f"{len(insight_topic_support_df):,} insight-topic support rows."
)

---
## Step 10 — Packaging, Dedupe & Topline Selection

Three sequential operations:

1. **Packaging** — apply quality threshold gates. Insights that don't clear
   `min_verified_topic_count`, `min_supporting_project_count`, and
   `min_mean_topic_share` are rejected.
2. **Deduplication** — deterministic overlap-based dedup within pair types.
3. **Topline selection** — assigns `report_section` to each accepted insight:
   `main_cross` (top N by quality rank), `main_by_group` (top 1 per group),
   `appendix_cross`, `appendix_by_group`.

In [17]:
# ── PACKAGING + DEDUPE + SIMPLE TOPLINE CUT ────────────────────────────────
# Business rule:
# - accept insights that clear threshold gates
# - remove obvious duplicates
# - then choose topline from the remaining pool
#
# Main section:
# - top N cross-category insights by:
#     1) supporting_project_count
#     2) verified_topic_count
#     3) mean_topic_share_all_verified_topics
# - top 1 by-group insight per category using the same ranking
#
# Appendix:
# - all other accepted insights

packaging = apply_deterministic_packaging(
    insights_flat_df,
    output_group_key=OUTPUT_GROUP_KEY,
    packaging_cfg=PACKAGING_CFG,
)

accepted_pack_df = packaging["accepted_df"]

print(f"Total insights before packaging: {len(insights_flat_df)}")
print(f"Accepted after thresholds: {len(accepted_pack_df)}")

curated_df, dedup_audit_df = dedupe_packaged_insights(
    accepted_pack_df,
    dedupe_cfg=DEDUPE_CFG,
)

curated_df = curated_df.copy()
curated_df["looker_url"] = curated_df["top_project_ids"].apply(
    lambda ids: build_looker_project_url(
        base_url=LOOKER_BASE_URL,
        project_ids=ids,
        filter_field=LOOKER_FILTER_FIELD,
        fields=LOOKER_FIELDS,
        limit=LOOKER_LIMIT,
        max_ids=LOOKER_ID_LIMIT,
    )
)

# main_min_verification_ratio gates eligibility for main-section placement;
# sourced from params.yaml → analysis.packaging.main_min_verification_ratio.
curated_df = assign_topline_sections_simple(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
    main_cross_limit=PACKAGING_CFG["main_cross_limit"],
    main_min_verification_ratio=MAIN_MIN_VERIFICATION_RATIO,
)

main_cross_df = curated_df[curated_df["report_section"] == "main_cross"].copy()
main_by_group_df = curated_df[curated_df["report_section"] == "main_by_group"].copy()
appendix_df = curated_df[
    curated_df["report_section"].isin(["appendix_cross", "appendix_by_group"])
].copy()

accepted_ids = set(curated_df["insight_id"])
rejected_ids = set(insights_flat_df["insight_id"]) - accepted_ids

# export project_id list by insight
insight_project_df = (
    insight_topic_support_df[
        insight_topic_support_df["insight_id"].isin(accepted_ids)
    ][["insight_id", "group_value", "topic_id"]]
    .merge(
        project_topic_bridge_df[["project_id", GROUPBY_FIELD, "topic_id"]],
        left_on=["group_value", "topic_id"],
        right_on=[GROUPBY_FIELD, "topic_id"],
        how="left",
    )
    [["insight_id", "project_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
insight_project_df.to_csv(OUT("insights", "insight_project_bridge.csv"), index=False)

# ── EXPORT: insight-with-text + sampled essays ────────────────────────────
# Companion to insight_project_bridge.csv, but with:
#   1) insight title / finding / evidence fields instead of insight_id only
#   2) essay text instead of project_id only
#   3) capped at N projects per insight (random sample, fixed seed)
from utils import load_essay_snippet_lookup

ESSAY_SAMPLE_PER_INSIGHT = 20
ESSAY_MAX_CHARS = 1200  # bump if you want fuller essays per row

# 1) random-sample up to N projects per insight (deterministic via seed)
sampled_pairs_df = (
    insight_project_df
    .groupby("insight_id", group_keys=False)
    .apply(
        lambda g: g.sample(
            n=min(ESSAY_SAMPLE_PER_INSIGHT, len(g)),
            random_state=42,
        )
    )
    .reset_index(drop=True)
)

# 2) look up essay text for just the sampled project IDs
import re

ACTUAL_TEXT_COL = "tokens"
needed_ids = set(sampled_pairs_df["project_id"].unique())
essay_lookup: dict = {}

for fpath in sorted((ROOT / "DATA").glob("project_essays*.csv")):
    if len(essay_lookup) == len(needed_ids):
        break
    for chunk in pd.read_csv(
        fpath, usecols=["project_id", ACTUAL_TEXT_COL], chunksize=200_000
    ):
        sub = chunk[chunk["project_id"].isin(needed_ids - set(essay_lookup))]
        for _, row in sub.iterrows():
            text = re.sub(r"\s+", " ", str(row.get(ACTUAL_TEXT_COL, "") or "")).strip()
            if text:
                essay_lookup[row["project_id"]] = text[:ESSAY_MAX_CHARS]
        if len(essay_lookup) == len(needed_ids):
            break
sampled_pairs_df["essay_text"] = (
    sampled_pairs_df["project_id"].map(essay_lookup).fillna("")
)

# 3) attach the polished insight text from curated_df
insight_text_df = curated_df[[
    "insight_id", "title", "finding", "evidence_basis",
    "scope_or_caveat", "why_it_matters",
    "category_bucket", "report_section",
]].drop_duplicates(subset=["insight_id"])

insight_project_text_df = (
    sampled_pairs_df
    .merge(insight_text_df, on="insight_id", how="left")
    [[
        "insight_id", "report_section", "category_bucket",
        "title", "finding", "evidence_basis",
        "scope_or_caveat", "why_it_matters",
        "project_id", "essay_text",
    ]]
    .sort_values(["report_section", "insight_id", "project_id"])
    .reset_index(drop=True)
)

insight_project_text_df.to_csv(
    OUT("insights", "insight_project_text_sample.csv"), index=False
)
print(
    f"Wrote insight_project_text_sample.csv: "
    f"{len(insight_project_text_df):,} rows across "
    f"{insight_project_text_df['insight_id'].nunique()} insights "
    f"(up to {ESSAY_SAMPLE_PER_INSIGHT} essays each)"
)

curated_df.to_csv(OUT('chart_data', 'curated_df.csv'), index=False)

print(f"Accepted before dedupe: {len(accepted_pack_df)}")
print(f"Dropped as obvious duplicates: {len(dedup_audit_df)}")
print(f"Accepted after dedupe: {len(curated_df)}")
print(f"Main cross-category selected: {len(main_cross_df)}")
print(f"Main by-group selected: {len(main_by_group_df)}")
print(f"Appendix selected: {len(appendix_df)}")
print(f"{len(insight_project_df):,} insight-project pairs across {insight_project_df['insight_id'].nunique()} insights")

Total insights before packaging: 24
Accepted after thresholds: 15


/var/folders/j3/jwjf6cwj7czdz1klxbhhjst80000gp/T/ipykernel_20896/3042952426.py:94: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Wrote insight_project_text_sample.csv: 300 rows across 15 insights (up to 20 essays each)
Accepted before dedupe: 15
Dropped as obvious duplicates: 0
Accepted after dedupe: 15
Main cross-category selected: 5
Main by-group selected: 7
Appendix selected: 3
31,372 insight-project pairs across 15 insights


---
## Step 11 — Final Outputs & Manifests

Persists all accepted and rejected outputs, builds the DOCX report, and writes
stage and pipeline manifests. Running this cell again after a partial run is
safe — all writes are deterministic.

In [20]:
# ── FINAL OUTPUTS + REPORTING ──────────────────────────────────────────────
# Persist accepted/rejected outputs, evidence exports, chart data, DOCX, and manifests.

structured = build_structured_from_curated(
    curated_df,
    output_group_key=OUTPUT_GROUP_KEY,
)

# Add run metadata to structured insights for future multi-run HTML/review use.
def _add_run_meta_to_structured_item(item):
    item["strategic_loop_enabled"] = STRATEGIC_LOOP_ENABLED
    item["strategic_area_id"] = STRATEGIC_RUN_META.get("strategic_area_id")
    item["strategic_area_label"] = STRATEGIC_RUN_META.get("strategic_area_label")
    item["split_id"] = STRATEGIC_RUN_META.get("split_id")
    item["resolved_groupby_field"] = GROUPBY_FIELD
    return item

structured["key_insights"] = [
    _add_run_meta_to_structured_item(i)
    for i in structured.get("key_insights", [])
]
structured[OUTPUT_GROUP_KEY] = {
    group_value: [
        _add_run_meta_to_structured_item(i)
        for i in items
    ]
    for group_value, items in structured.get(OUTPUT_GROUP_KEY, {}).items()
}

write_json(OUT("insights", "insights_structured.json"), structured)

rejected_df = insights_flat_df[
    ~insights_flat_df["insight_id"].isin(curated_df["insight_id"])
].copy()
write_json(
    OUT("insights", "rejected_insights.json"),
    rejected_df.to_dict(orient="records"),
)

insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].to_csv(
    OUT("insights", "insight_topic_support.csv"),
    index=False,
)

chart_cols = [
    "run_id",
    "insight_id",
    "title",
    "section",
    "category_bucket",
    "supporting_project_count",
    "verified_topic_count",
    "mean_topic_share_all_verified_topics",
    "verification_ratio",
    "report_section",
    "report_order",
    "strategic_loop_enabled",
    "strategic_area_id",
    "strategic_area_label",
    "split_id",
    "resolved_groupby_field",
]
chart_ready_insights_df = curated_df[
    [c for c in chart_cols if c in curated_df.columns]
].rename(columns={"title": "theme"})

chart_ready_group_support_df = insight_topic_support_df[
    insight_topic_support_df["insight_id"].isin(curated_df["insight_id"])
].copy()

chart_ready_insights_df.to_csv(
    OUT("chart_data", "chart_ready_insights.csv"),
    index=False,
)
chart_ready_group_support_df.to_csv(
    OUT("chart_data", "chart_ready_group_support.csv"),
    index=False,
)

docx_path = OUT("reports", "insights_report.docx")
build_packaged_report_docx(
    structured=structured,
    output_path=docx_path,
    output_group_key=OUTPUT_GROUP_KEY,
    report_cfg=CFG["output"],
    project_count=df["project_id"].nunique(),
    run_id=RUN_ID,
)


# Persist LLM usage / cache / Flex telemetry for cost and reliability review.
llm_call_log_path = OUT("metadata", "llm_call_log.csv")
if "LLM_CALL_LOG_ROWS" in globals() and LLM_CALL_LOG_ROWS:
    llm_call_log_df = pd.DataFrame(LLM_CALL_LOG_ROWS)
    llm_call_log_df.to_csv(llm_call_log_path, index=False)

    llm_call_summary = (
        llm_call_log_df
        .groupby(["call_family", "call_name"], dropna=False)
        .agg(
            calls=("call_family", "size"),
            total_elapsed_sec=("elapsed_sec", "sum"),
            prompt_tokens=("prompt_tokens", "sum"),
            cached_tokens=("cached_tokens", "sum"),
            completion_tokens=("completion_tokens", "sum"),
            total_tokens=("total_tokens", "sum"),
            fallbacks=("fallback_used", "sum"),
        )
        .reset_index()
    )
    llm_call_summary["cached_share"] = (
        llm_call_summary["cached_tokens"] / llm_call_summary["prompt_tokens"]
    ).round(4)
    llm_call_summary.to_csv(OUT("metadata", "llm_call_summary.csv"), index=False)

    print("LLM call summary:")
    display(llm_call_summary)
else:
    pd.DataFrame().to_csv(llm_call_log_path, index=False)
    print("No logged LLM calls found; wrote empty llm_call_log.csv.")

groups_failed = sorted(set(NMF_GROUPS_FAILED) | set(LABELING_FAILED_GROUPS) | set(SYNTHESIS_FAILED_GROUPS))
eligible_groups = list(per_group_results.keys())
manifest_status = "failure" if groups_failed else "success"

stage_manifest_path = OUT("metadata", "stage_manifest_03_insights_generation.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status=manifest_status,
    input_artifacts=[
        artifact_meta(ROOT / "OUTPUTS/prepared/06_enriched.parquet", "enriched_parquet"),
        artifact_meta(ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json", "stage_manifest_01"),
        artifact_meta(ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json", "stage_manifest_02"),
    ],
    output_artifacts=[
        artifact_meta(COPIED_CONFIG_PATH, "resolved_params_yaml"),
        artifact_meta(FILTER_SPEC_PATH, "filter_spec_json"),
        artifact_meta(FILTER_SUMMARY_PATH, "filter_summary_json"),
        artifact_meta(STRATEGIC_META_PATH, "strategic_run_meta_json"),
        artifact_meta(OUT("analysis", "category_tfidf.csv"), "category_tfidf_csv"),
        artifact_meta(OUT("analysis", "nmf_topics.csv"), "nmf_topics_csv"),
        artifact_meta(OUT("analysis", "nmf_weights.csv"), "nmf_weights_csv"),
        artifact_meta(OUT("analysis", "project_topic_bridge.csv"), "project_topic_bridge_csv"),
        artifact_meta(OUT("analysis", "llm_topic_labels.json"), "llm_topic_labels_json"),
        artifact_meta(OUT("insights", "insights_candidates_draft_pre_metadata.json"), "insights_candidates_draft_pre_metadata_json"),
        artifact_meta(OUT("insights", "metadata_lift_facts_for_step7.csv"), "metadata_lift_facts_for_step7_csv"),
        artifact_meta(OUT("insights", "insights_candidates.json"), "insights_candidates_json"),
        artifact_meta(OUT("insights", "insights_structured.json"), "insights_structured_json"),
        artifact_meta(OUT("insights", "rejected_insights.json"), "rejected_insights_json"),
        artifact_meta(OUT("insights", "insights_flat.csv"), "insights_flat_csv"),
        artifact_meta(OUT("insights", "insight_topic_support.csv"), "insight_topic_support_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_insights.csv"), "chart_ready_insights_csv"),
        artifact_meta(OUT("chart_data", "chart_ready_group_support.csv"), "chart_ready_group_support_csv"),
        artifact_meta(OUT("reports", "insights_report.docx"), "insights_report_docx"),
    ],
    row_counts={
        "input_rows": int(len(raw_df)),
        "input_projects": int(raw_df["project_id"].nunique()),
        "filtered_projects_before_strategic_prep": int(BASE_FILTERED_PROJECT_COUNT),
        "run_rows": int(len(df)),
        "run_projects": int(df["project_id"].nunique()),
        "eligible_groups": int(len(eligible_groups)),
        "groups_skipped": int(len(NMF_GROUPS_SKIPPED)),
        "groups_failed": int(len(groups_failed)),
        # v1.4-compatible keys. DEPRECATED after NB04 and audit readers move to v1.5 aliases.
        "topics_generated": int(len(topics_df)),
        "insights_generated": int(len(insights_flat_df)),
        "insights_accepted": int(len(accepted_ids)),
        "insights_rejected": int(len(rejected_ids)),
        # v1.5 aliases
        "topics": int(len(topics_df)),
        "candidate_insights": int(len(insights_flat_df)),
        "accepted_insights": int(len(curated_df)),
        "rejected_insights": int(len(rejected_ids)),
        "metadata_lift_facts_for_step7": int(len(metadata_lift_audit_df)),
    },
    key_params={
        "groupby_field": GROUPBY_FIELD,
        "base_groupby_field": BASE_GROUPBY_FIELD,
        "strategic_loop_enabled": STRATEGIC_LOOP_ENABLED,
        "strategic_run_meta": STRATEGIC_RUN_META,
        "topic_assignment_threshold": CFG["analysis"]["topic_assignment_threshold"],
        "verification_config": VERIFY_CFG,
        "packaging_config": PACKAGING_CFG,
        "dedupe_config": DEDUPE_CFG,
        "metadata_lift_enabled": METADATA_LIFT_ENABLED,
        "metadata_lift_dimensions": METADATA_LIFT_DIMENSIONS,
        "metadata_lift_thresholds": METADATA_LIFT_THRESHOLDS,
        "min_group_projects_effective": MIN_GROUP_PROJECTS_EFFECTIVE,
        "small_slice_mode": SMALL_SLICE_MODE,
        "models": {
            "labeling": MODEL_LABELING,
            "synthesis": MODEL_SYNTHESIS,
            "verify": MODEL_VERIFY,
        },
    },
    warnings_path=WARNINGS_PATH,
)

pipeline_manifest_path = OUT("metadata", "pipeline_manifest.json")
build_pipeline_manifest(
    output_path=pipeline_manifest_path,
    run_id=RUN_ID,
    run_date=RUN_DATE,
    group_by_field=GROUPBY_FIELD,
    filter_spec_path=FILTER_SPEC_PATH,
    filter_summary_path=FILTER_SUMMARY_PATH,
    stage_manifest_paths=[
        ROOT / "OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json",
        ROOT / "OUTPUTS/enrichment/metadata/stage_manifest_02_semantic_enrichment.json",
        stage_manifest_path,
    ],
    warnings_01_path=ROOT / "OUTPUTS/prepared/metadata/warnings_01.jsonl",
    warnings_02_path=ROOT / "OUTPUTS/enrichment/metadata/warnings_02.jsonl",
    warnings_03_path=WARNINGS_PATH,
    final_outputs={
        # v1.4-compatible keys. DEPRECATED after NB04 and audit readers move to v1.5 aliases.
        "insights_structured_json": str(OUT("insights", "insights_structured.json")),
        "insights_flat_csv": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support_csv": str(OUT("insights", "insight_topic_support.csv")),
        "chart_ready_insights_csv": str(OUT("chart_data", "chart_ready_insights.csv")),
        "chart_ready_group_support_csv": str(OUT("chart_data", "chart_ready_group_support.csv")),
        "insights_report_docx": str(docx_path),
        "llm_call_log_csv": str(OUT("metadata", "llm_call_log.csv")),
        "llm_call_summary_csv": str(OUT("metadata", "llm_call_summary.csv")),
        # v1.5 additions / aliases
        "metadata_lift_facts_for_step7_csv": str(OUT("insights", "metadata_lift_facts_for_step7.csv")),
        "insights_structured": str(OUT("insights", "insights_structured.json")),
        "insights_flat": str(OUT("insights", "insights_flat.csv")),
        "insight_topic_support": str(OUT("insights", "insight_topic_support.csv")),
        "metadata_lift_facts_for_step7": str(OUT("insights", "metadata_lift_facts_for_step7.csv")),
        "insights_report": str(docx_path),
    },
    config_path=MANIFEST_CFG_PATH,
    filter_fields_key=FILTER_FIELDS_KEY,
    status=manifest_status,
)

print(f"Final structured insights: {OUT('insights', 'insights_structured.json')}")
print(f"DOCX report: {docx_path}")
print(f"Stage manifest: {stage_manifest_path}")
print(f"Pipeline manifest: {pipeline_manifest_path}")
print(f"Manifest status: {manifest_status}")
if groups_failed:
    print(f"Groups with partial failures: {groups_failed}")


LLM call summary:


,call_family,call_name,calls,total_elapsed_sec,prompt_tokens,cached_tokens,completion_tokens,total_tokens,fallbacks,cached_share
0,insights_draft,insights_draft,1,79.101,21753,0,4807,26560,0,0.0000
1,insights_finalize,insights_finalize,1,56.970,13499,0,5572,19071,0,0.0000
2,synthesis,synthesis,10,105.471,40334,2560,10018,50352,0,0.0635


Final structured insights: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/strategic_area/stem/20260522_160015_strategic_injected_tag_b09019b8/insights/insights_structured.json
DOCX report: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/strategic_area/stem/20260522_160015_strategic_injected_tag_b09019b8/reports/trend_tracker_report.docx
Stage manifest: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/strategic_area/stem/20260522_160015_strategic_injected_tag_b09019b8/metadata/stage_manifest_03_insights_generation.json
Pipeline manifest: /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/runs/strategic_area/stem/20260522_160015_strategic_injected_tag_b09019b8/metadata/pipeline_manifest.json
Manifest status: success
